# 04 — Prétraitement

Ce notebook explique et justifie les choix de prétraitement de House Prices. Les règles réutilisables sont importées depuis `src/preprocessing/`. Nous partons des anomalies et des distributions observées dans le [notebook 03, analyse exploratoire](03_analyse_exploratoire.ipynb) (étape 3).

## 1. Protocole

Dans les CSV, `NA` signale une valeur absente. Le mot `None` est un code valide, notamment pour une maison sans parement de maçonnerie. Nous désactivons donc les autres marqueurs de valeurs manquantes.

Pour comparer les choix de prétraitement, nous utilisons une validation croisée à cinq plis sur les 1 460 ventes étiquetées (`random_state=42`). Chaque vente sert tour à tour à l’entraînement et à l’évaluation : les comparaisons profitent de toutes les données, ce qui réduit leur bruit. Tout paramètre appris, dont les médianes et les coefficients du modèle, provient des seules maisons d’entraînement du pli. Le fichier Kaggle `test.csv` sert ici à compter et illustrer les corrections.

**Ventes typiques.** Les deux ventes aberrantes (maisons 524 et 1299) sont retirées de l’entraînement de chaque pli, comme le suggère l’auteur du jeu de données, De Cock (2011). La section 2.6 justifie ce choix. Elles sont aussi exclues du calcul des scores : leurs erreurs, très grandes, domineraient la comparaison.

In [1]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    FunctionTransformer, OneHotEncoder, StandardScaler,
)

from src.evaluation import cv_rmse, rmse_log
from src.preprocessing.steps.cleaning import Cleaner, LARGE_HOUSE_AREA, remove_outliers
from src.preprocessing.steps.encoding import (
    ORDINAL_COLUMNS, QUALITY_COLUMNS, RARE_CATEGORY_MIN_COUNT, QualityOrdinalEncoder,
    encoding_summary, make_onehot_encoder, nominal_columns,
)
from src.preprocessing.steps.features import AGE_FEATURES, SUM_FEATURES, FeatureCreator
from src.preprocessing.steps.missing import (
    CASE_NAMES, EQUIPMENT_GROUPS, NO_MEASURE_ABSENCE, MissingValueHandler, consistency_report,
    imputation_report,
)
from src.preprocessing.steps.scaling import SkewCorrector, make_scaler
from src.preprocessing.pipelines import (
    RECIPES, make_base, make_causal, make_causal_pca, make_correlation, make_pca,
)
from src.preprocessing.strategies.causal import CAUSAL_MODE, CONSEQUENCES, DUPLICATES, causal_drops


def show_title(title):
    """Afficher dans les sorties le titre de la sous-section correspondante du texte."""
    display(Markdown(f"**{title}**"))

# ──────────────────────────────────────────────────────────────────────
# 1 · Données et plis
# ──────────────────────────────────────────────────────────────────────
show_title("1 · Données et plis")
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
read_options = {"na_values": ["NA"], "keep_default_na": False}
train = pd.read_csv(root / "data/train.csv", **read_options)
test = pd.read_csv(root / "data/test.csv", **read_options)
X, y = train.drop(columns="SalePrice"), train["SalePrice"]
folds = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"{len(X)} ventes étiquetées en validation croisée à {folds.n_splits} plis")
print(f"{len(test)} maisons dans test.csv.")

**1 · Données et plis**

1460 ventes étiquetées en validation croisée à 5 plis
1459 maisons dans test.csv.


## 2. Nettoyage

Nous corrigeons les erreurs manifestes et marquons comme inconnue toute information contradictoire dont la valeur correcte nous échappe. L’imputation relève de la section 3. Une catégorie rare peut encore décrire des maisons différentes. La section 7 évaluera l’intérêt prédictif des variables conservées.

### 2.0 Doublons et identifiant

Un identifiant unique pourrait masquer des doublons descriptifs. Nous vérifions séparément les identifiants, les lignes complètes et les caractéristiques seules.

**Résultat**: Aucun identifiant ni aucune ligne complète ne se répète dans train. Nous gardons les ventes et retirons `Id` des prédicteurs. Nous le conservons dans les tableaux descriptifs pour retrouver chaque maison.

In [2]:
# ──────────────────────────────────────────────────────────────────────
# 2.0 · Doublons et identifiant
# ──────────────────────────────────────────────────────────────────────
show_title("2.0 · Doublons et identifiant")
display(pd.Series({
    "Identifiants répétés": train["Id"].duplicated().sum(),
    "Lignes identiques": train.duplicated().sum(),
    "Caractéristiques identiques hors Id et prix": X.drop(columns="Id").duplicated().sum(),
}, name="Répétitions dans train"))
cleaner = Cleaner().fit(X)
raw = pd.concat({"train": X, "test": test}, names=["Fichier", "Ligne"])
clean = pd.concat({"train": cleaner.transform(X), "test": cleaner.transform(test)},
                  names=raw.index.names)

def rule_doc(column, method=Cleaner.transform):
    """Afficher la ligne de documentation correspondant à la règle étudiée."""
    lines = inspect.getdoc(method).splitlines()
    print(f"{method.__qualname__} —", next(line for line in lines if line.startswith(column + " ")))

def changed(column):
    """Repérer une correction, en considérant deux valeurs manquantes comme identiques."""
    before, after = raw[column], clean[column]
    same = before.eq(after).fillna(False) | (before.isna() & after.isna())
    return ~same

**2.0 · Doublons et identifiant**

Identifiants répétés                           0
Lignes identiques                              0
Caractéristiques identiques hors Id et prix    0
Name: Répétitions dans train, dtype: int64

### 2.1 Type de logement

**Problème.** Dans le notebook 03 (section 1, « Description des données »), `MSSubClass` est lu comme un entier. Dans ce notebook-ci, la sortie « 2.1 Type de logement » ci-dessous regroupe toutes les maisons concernées par code et par fichier.

**Décision.** Convertir le code en texte pour traiter `MSSubClass` comme une variable catégorielle. 

**Code.** Règle `MSSubClass` de `Cleaner.transform`.

**Justification.** Le code 20 désigne une maison à un étage construite à partir de 1946, le code 60 une maison à deux étages de la même période, et le code 190 une conversion bifamiliale. Un coefficient appliqué au nombre imposerait un ordre et une distance arbitraires. L’encodage catégoriel pourra aussi traiter le code 150, observé seulement dans test.

In [3]:
# ──────────────────────────────────────────────────────────────────────
# 2.1 Type de logement
# ──────────────────────────────────────────────────────────────────────
show_title("2.1 Type de logement")
rule_doc("MSSubClass")
display(pd.crosstab(raw["MSSubClass"], raw.index.get_level_values("Fichier")))

**2.1 Type de logement**

Cleaner.transform — MSSubClass est convertie en texte pour être traitée comme une variable catégorielle.


col_0,test,train
MSSubClass,,
20,543,536
30,70,69
40,2,4
45,6,12
50,143,144
60,276,299
70,68,60
75,7,16
80,60,58


### 2.2 Revêtements extérieurs

**Problème.** `Exterior1st` indique le revêtement principal des murs et `Exterior2nd` le revêtement secondaire. Le notebook 03 (section 1, « Description des données ») relève trois différences d’orthographe entre ces colonnes.

**Décision.** Corriger l’orthographe de trois codes de `Exterior2nd` pour qu’ils s’écrivent comme dans `Exterior1st` et dans le dictionnaire. 

**Code.** Règle `Exterior2nd` de `Cleaner.transform`.

**Justification.** Une maison entièrement en bardeaux de bois pouvait porter à la fois `WdShing` et `Wd Shng`. Une comparaison directe aurait alors compté deux matériaux. La correction conserve l’information sur le matériau.

Les codes de `BldgType`, `MSZoning` et `Neighborhood` s’écartent aussi du dictionnaire (notebook 03, section 1), par exemple `Duplex` au lieu de `Duplx`. Chacun s’écrit toutefois de la même façon dans train et dans test : l’encodage reconnaît donc les mêmes catégories, et nous les laissons tels quels.

In [4]:
# ──────────────────────────────────────────────────────────────────────
# 2.2 Revêtements extérieurs
# ──────────────────────────────────────────────────────────────────────
show_title("2.2 Revêtements extérieurs")
rule_doc("Exterior2nd")
exterior = raw.loc[changed("Exterior2nd"), ["Id", "Exterior1st", "Exterior2nd"]].copy()
exterior["Code harmonisé"] = clean.loc[exterior.index, "Exterior2nd"]
display(exterior.groupby(["Exterior2nd", "Code harmonisé", "Fichier"]).size()
        .unstack("Fichier", fill_value=0))
display(exterior)

**2.2 Revêtements extérieurs**

Cleaner.transform — Exterior2nd corrige trois codes pour les écrire comme dans Exterior1st.


,Fichier,test,train
Exterior2nd,Code harmonisé,,
Brk Cmn,BrkComm,15,7
CmentBd,CemntBd,66,60
Wd Shng,WdShing,43,38


Id Exterior1st Exterior2nd Code harmonisé
Fichier Ligne                                             
train   3         4     Wd Sdng     Wd Shng        WdShing
        8         9     BrkFace     Wd Shng        WdShing
        11       12     WdShing     Wd Shng        WdShing
        23       24     CemntBd     CmentBd        CemntBd
        53       54     WdShing     Wd Shng        WdShing
...             ...         ...         ...            ...
test    1452   2913     CemntBd     CmentBd        CemntBd
        1453   2914     CemntBd     CmentBd        CemntBd
        1454   2915     CemntBd     CmentBd        CemntBd
        1455   2916     CemntBd     CmentBd        CemntBd
        1457   2918     HdBoard     Wd Shng        WdShing

[229 rows x 4 columns]

### 2.3 Services publics et variables quasi constantes

**Problème.** Le notebook 03 (section 2, « Statistiques descriptives », et synthèse de la section 7) signale sept variables quasi constantes : une seule valeur y couvre au moins 95 % des maisons de train. Une telle variable n’est utile que si ses rares exceptions renseignent sur les maisons à prédire.

**Vérification.** La sortie « 2.3 Exceptions des variables quasi constantes » ci-dessous donne, pour chacune, la valeur majoritaire de train, le nombre de maisons qui s’en écartent dans train et dans test, et le prix médian des deux groupes dans train. Pour `PoolQC` et `MiscFeature`, `NA` signifie un équipement absent; pour `Utilities`, une valeur inconnue, comptée à part.

In [5]:
# ──────────────────────────────────────────────────────────────────────
# 2.3 Exceptions des variables quasi constantes
# ──────────────────────────────────────────────────────────────────────
show_title("2.3 Exceptions des variables quasi constantes")
quasi_constant = ["Utilities", "Street", "PoolQC", "Condition2", "RoofMatl", "Heating", "MiscFeature"]
# Pour ces deux variables, NA signifie un équipement absent (dictionnaire des variables).
absence_columns = {"PoolQC", "MiscFeature"}

def known_values(frame, column):
    """Remplacer NA par « Absent » quand il signifie un équipement absent; sinon le laisser inconnu."""
    return frame[column].fillna("Absent") if column in absence_columns else frame[column]

rows = []
for column in quasi_constant:
    in_train, in_test = known_values(train, column), known_values(test, column)
    majority = in_train.mode().iat[0]
    unusual = in_train.notna() & in_train.ne(majority)
    rows.append([column, majority, unusual.sum(), (in_test.notna() & in_test.ne(majority)).sum(),
                 in_test.isna().sum(), y[in_train.eq(majority)].median(), y[unusual].median()])
dollars = lambda value: f"{value:,.0f} $".replace(",", " ")
display(pd.DataFrame(rows, columns=["Variable", "Majorité train", "Exceptions train", "Exceptions test",
                                     "Inconnues test", "Prix médian majorité", "Prix médian exceptions"])
        .style.hide(axis="index")
        .format({"Prix médian majorité": dollars, "Prix médian exceptions": dollars}))

**2.3 Exceptions des variables quasi constantes**

Variable,Majorité train,Exceptions train,Exceptions test,Inconnues test,Prix médian majorité,Prix médian exceptions
Utilities,AllPub,1,0,2,163 000 $,137 500 $
Street,Pave,6,6,0,163 000 $,114 250 $
PoolQC,Absent,7,3,0,162 900 $,235 000 $
Condition2,Norm,15,15,0,163 500 $,128 000 $
RoofMatl,CompShg,26,17,0,162 000 $,234 750 $
Heating,GasA,32,13,0,164 500 $,116 750 $
MiscFeature,Absent,54,51,0,164 250 $,146 000 $


**Lecture.** `Utilities` se distingue. Une seule maison de train a la modalité `NoSeWa` (137 500 \$), et aucune maison de test ne s’écarte de `AllPub` (deux valeurs y sont inconnues). La colonne est constante pour les maisons à prédire : elle ne peut rien apporter, et une seule exception ne permet pas d’estimer un effet.

Les six autres variables ont des exceptions dans les deux fichiers (de 3 à 51 maisons de test), et leur prix médian s’écarte nettement de celui de la majorité : 114 250 $ pour les 6 maisons sur rue en gravier (`Street`) contre 163 000 \$, ou 235 000 \$ pour les 7 maisons avec piscine (`PoolQC`) contre 162 900 \$. Ces écarts reposent sur peu de maisons, mais ces variables peuvent aider à prédire quelques maisons de test.

**Décision.** Retirer `Utilities` et conserver les six autres variables. Leurs modalités rares seront regroupées à l’encodage (section 5.4). 

**Code.** Règle `Utilities` de `Cleaner.transform`.

In [6]:
# ──────────────────────────────────────────────────────────────────────
# 2.3 Retrait de Utilities
# ──────────────────────────────────────────────────────────────────────
show_title("2.3 Retrait de Utilities")
rule_doc("Utilities")
kept = [column for column in quasi_constant if column in clean.columns]
print(f"Utilities présente après nettoyage : {'Utilities' in clean.columns}. "
      f"Variables conservées : {', '.join(kept)}.")

**2.3 Retrait de Utilities**

Cleaner.transform — Utilities est écartée car sa seule exception connue est dans train.
Utilities présente après nettoyage : False. Variables conservées : Street, PoolQC, Condition2, RoofMatl, Heating, MiscFeature.


### 2.4 Années du garage et de la rénovation

**Problème.** Deux garages de test ont une année postérieure à la vente (notebook 03, section 3, « Identification des données manquantes »). La sortie « 2.4 Année du garage » ci-dessous rapproche l’année déclarée du type de garage et des dates de la maison.

**Décision.** Si l’année du garage dépasse celle de la vente de plus d’un an, reprendre l’année de rénovation de cette même maison. 

**Code.** Règle `GarageYrBlt` de `Cleaner.transform`.

**Justification.** Les ventes vont de 2006 à 2010. 

Maison 2593: L’année 2207 du garage attaché est une faute de frappe manifeste. Sa rénovation date de 2007, année plausible que la règle retrouve sans consulter l’identifiant. Kaggle exige une prédiction pour chaque maison de test. Garder la maison, l’existence de son garage et une année approximative préserve l’information disponible. 

Maison 2550: Une vente sur plan en 2007 est compatible avec une construction en 2008. La tolérance d’un an conserve cette date.

In [7]:
# ──────────────────────────────────────────────────────────────────────
# 2.4 Année du garage
# ──────────────────────────────────────────────────────────────────────
show_title("2.4 Année du garage")
rule_doc("GarageYrBlt")
garage = raw.loc[raw["GarageYrBlt"].gt(raw["YrSold"]),
                 ["Id", "GarageType", "GarageYrBlt", "YearBuilt", "YearRemodAdd", "YrSold"]].copy()
garage["Année retenue"] = clean.loc[garage.index, "GarageYrBlt"]
display(garage)

**2.4 Année du garage**

Cleaner.transform — GarageYrBlt reprend YearRemodAdd si la date dépasse la tolérance après vente.


Id GarageType  GarageYrBlt  YearBuilt  YearRemodAdd  YrSold  \
Fichier Ligne                                                                  
test    1089   2550     Attchd       2008.0       2008          2009    2007   
        1132   2593     Attchd       2207.0       2006          2007    2007   

               Année retenue  
Fichier Ligne                 
test    1089          2008.0  
        1132          2007.0

**Problème.** La vérification de cohérence du notebook 03 (section 3) signale aussi une maison de test, la 1877, dont la rénovation (2001) précède la construction (2002). La sortie « 2.4 Année de rénovation » ci-dessous la montre.

**Décision.** Si `YearRemodAdd` précède `YearBuilt`, reprendre l’année de construction. 

**Code.** Règle `YearRemodAdd` de `Cleaner.transform`.

**Justification.** Selon le dictionnaire, `YearRemodAdd` égale `YearBuilt` lorsque la maison n’a pas été rénovée. Une rénovation antérieure à la construction est impossible. L’écart d’un an évoque une saisie approximative pour une maison jamais rénovée. La correction évite un temps depuis la rénovation (8 ans) plus long que l’âge de la maison (7 ans).

In [8]:
# ──────────────────────────────────────────────────────────────────────
# 2.4 Année de rénovation
# ──────────────────────────────────────────────────────────────────────
show_title("2.4 Année de rénovation")
rule_doc("YearRemodAdd")
remodel = raw.loc[raw["YearRemodAdd"].lt(raw["YearBuilt"]),
                  ["Id", "YearBuilt", "YearRemodAdd", "YrSold"]].copy()
remodel["Année retenue"] = clean.loc[remodel.index, "YearRemodAdd"]
display(remodel)

**2.4 Année de rénovation**

Cleaner.transform — YearRemodAdd reprend YearBuilt si la rénovation précède la construction.


,,Id,YearBuilt,YearRemodAdd,YrSold,Année retenue
Fichier,Ligne,,,,,
test,416,1877,2002,2001,2009,2002


### 2.5 Parement de maçonnerie

**Problème.** Le parement est une couche décorative de brique ou de pierre sur une partie des murs extérieurs. `MasVnrType` décrit le matériau et `MasVnrArea` la surface en pi². Le notebook 03 (section 3, « Identification des données manquantes ») signale des contradictions entre les deux. Pour situer leur ordre de grandeur, nous calculons la surface médiane des parements de surface positive et de matériau connu dans train.

**Décision.** Corriger la surface bouche-trou. Pour les autres contradictions, marquer le seul champ douteux comme inconnu. 

**Code.** Règle `MasVnrType` de `Cleaner.transform`.

**Justification.** Un parement couvre typiquement environ 200 pi² (médiane des parements renseignés de train : 203 pi²). Une surface d’un pied carré pour un parement déclaré absent est une valeur bouche-trou. À l’inverse, plusieurs centaines de pieds carrés rendent le matériau déclaré absent suspect. Quand le matériau est connu et que la surface vaut zéro, c’est la mesure de surface qui manque.

| Cas | Id dans train | Id dans test | Champ probablement faux et correction |
|---|---|---|---|
| Aucun parement, 1 pi² | 774, 1231 | 2453 | Surface bouche-trou ramenée à 0 |
| Aucun parement, surface supérieure à 1 pi² | 625, 1301, 1335 | 1670 | Type inconnu, surface conservée |
| Brique ou pierre, 0 pi² | 689, 1242 | 2320 | Surface inconnue, matériau conservé |

La section 3 devra distinguer ces inconnues créées par le nettoyage des `NA` d’origine lors de l’imputation.

In [9]:
# ──────────────────────────────────────────────────────────────────────
# 2.5 Parement de maçonnerie
# ──────────────────────────────────────────────────────────────────────
show_title("2.5 Parement de maçonnerie")
rule_doc("MasVnrType")
known = train["MasVnrType"].isin(["BrkFace", "BrkCmn", "Stone"]) & train["MasVnrArea"].gt(0)
print(f"Surface médiane des parements renseignés : {train.loc[known, 'MasVnrArea'].median():g} pi²")
area_changed = changed("MasVnrArea")
type_changed = changed("MasVnrType")
placeholder = area_changed & clean["MasVnrArea"].eq(0)
unknown_area = area_changed & clean["MasVnrArea"].isna()
masonry = raw.loc[type_changed | area_changed, ["Id", "MasVnrType", "MasVnrArea"]].copy()
masonry["Type retenu"] = clean.loc[masonry.index, "MasVnrType"]
masonry["Surface retenue"] = clean.loc[masonry.index, "MasVnrArea"]
display(masonry)

**2.5 Parement de maçonnerie**

Cleaner.transform — MasVnrType et MasVnrArea sont réconciliés selon la présence et la surface.
Surface médiane des parements renseignés : 203 pi²


Id MasVnrType  MasVnrArea Type retenu  Surface retenue
Fichier Ligne                                                          
train   624     625       None       288.0         NaN            288.0
        688     689    BrkFace         0.0     BrkFace              NaN
        773     774       None         1.0        None              0.0
        1230   1231       None         1.0        None              0.0
        1241   1242      Stone         0.0       Stone              NaN
        1300   1301       None       344.0         NaN            344.0
        1334   1335       None       312.0         NaN            312.0
test    209    1670       None       285.0         NaN            285.0
        859    2320    BrkFace         0.0     BrkFace              NaN
        992    2453       None         1.0        None              0.0

### 2.6 Valeurs aberrantes

Une vente est aberrante lorsque son prix paraît incohérent avec les caractéristiques de la maison. Un prix élevé peut très bien correspondre à une vaste maison de grande qualité. Nous reprenons les quatre grandes maisons du notebook 03 : tableau des très grandes maisons (section 4) et nuage de points (a) (section 6, « Visualisations pertinentes »).

Les maisons 524 et 1299, de qualité 10 à Edwards, se sont vendues moins de 185 000 \$ en condition `Partial`. Les deux grandes maisons de NoRidge se sont vendues beaucoup plus cher. 

Nous comparons quatre options pour l'entraînement : 
- A) Tout garder
- B) Retirer seulement les maisons de plus de 4 000 pi² vendues moins de 300 000 \$, c'est-à-dire 524 et 1299 (« retrait ciblé »)
- C) Retirer toutes les maisons de plus de 4 000 pi², comme le conseille De Cock, y compris les deux de NoRidge
- D) Retirer le 1 % des ventes les plus chères.

In [10]:
# ──────────────────────────────────────────────────────────────────────
# 2.6 Valeurs aberrantes · grandes maisons
# ──────────────────────────────────────────────────────────────────────
show_title("2.6 Valeurs aberrantes · grandes maisons")
display(train.loc[X["GrLivArea"].gt(LARGE_HOUSE_AREA),
                  ["Id", "Neighborhood", "GrLivArea", "OverallQual", "SaleCondition", "SalePrice"]])
print(inspect.getdoc(remove_outliers))

**2.6 Valeurs aberrantes · grandes maisons**

,Id,Neighborhood,GrLivArea,OverallQual,SaleCondition,SalePrice
523,524,Edwards,4676,10,Partial,184750
691,692,NoRidge,4316,10,Normal,755000
1182,1183,NoRidge,4476,10,Abnorml,745000
1298,1299,Edwards,5642,10,Partial,160000


Retirer les ventes incohérentes de très grandes maisons à faible prix.


**Comparaison provisoire.** Le prétraitement complet n’est construit qu’aux sections 3 à 8. Pour comparer les quatre options, nous utilisons donc un modèle simple : `Cleaner`, imputation médiane, `log1p` des surfaces, standardisation, encodage one-hot, puis une régression ridge (RidgeCV) sur le logarithme du prix. Seul le traitement des ventes aberrantes change d’une option à l’autre.

Dans chaque pli, les maisons sont retirées de l’entraînement seulement, avant l’ajustement. Le seuil du 1 % est calculé sur les prix d’entraînement du pli. Nous mesurons la RMSE des logarithmes sur trois groupes : toutes les ventes, les ventes typiques (sans 524 et 1299, section 1) et les 5 % de ventes les plus chères, pour voir l’effet de chaque option sur le haut de gamme.

In [11]:
# ──────────────────────────────────────────────────────────────────────
# 2.6 Valeurs aberrantes · modèle provisoire
# ──────────────────────────────────────────────────────────────────────
# Les surfaces sont exprimées en pi² dans le dictionnaire d'Ames.
area_columns = [
    "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "GarageArea", "WoodDeckSF",
    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
]

def other_numeric(frame):
    """Les autres grandeurs restent sur leur échelle avant standardisation."""
    return frame.select_dtypes(include="number").columns.difference(area_columns).tolist()

def add_garage_flag(frame):
    """Distinguer la présence du garage pour comparer ses années imputées."""
    return frame.assign(HasGarage=frame["GarageArea"].gt(0).astype(int))

def provisional_model(imputer=None):
    """Reprendre les mêmes transformations et le même modèle dans les deux comparaisons."""
    steps = [Cleaner()]
    area_steps = [FunctionTransformer(np.log1p), StandardScaler()]
    numeric = StandardScaler()
    nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if imputer is None:
        areas = make_pipeline(SimpleImputer(strategy="median"), *area_steps)
        numeric = make_pipeline(SimpleImputer(strategy="median"), numeric)
        nominal = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), nominal)
    else:
        steps.extend([imputer, FunctionTransformer(add_garage_flag)])
        areas = make_pipeline(*area_steps)
    columns = ColumnTransformer([
        ("areas", areas, area_columns), ("numeric", numeric, other_numeric),
        ("nominal", nominal, make_column_selector(dtype_exclude="number")),
    ])
    return make_pipeline(*steps, columns, RidgeCV(alphas=np.logspace(-1, 3, 30)))

In [12]:
# ──────────────────────────────────────────────────────────────────────
# 2.6 Valeurs aberrantes · comparaison des retraits
# ──────────────────────────────────────────────────────────────────────
show_title("2.6 Valeurs aberrantes · comparaison des retraits")
options = ["A · Tout garder", "B · Retrait ciblé", "C · Surface > 4 000 pi²", "D · Prix du centile supérieur"]
scores = []
for option in options:
    predictions = pd.Series(np.nan, index=y.index)
    removed = []
    for fit_index, valid_index in folds.split(X):
        features, prices = X.iloc[fit_index], y.iloc[fit_index]
        if option == options[1]:
            features, prices = remove_outliers(features, prices)
        elif option == options[2]:
            keep = features["GrLivArea"].le(LARGE_HOUSE_AREA)
            features, prices = features.loc[keep], prices.loc[keep]
        elif option == options[3]:
            keep = prices.le(prices.quantile(0.99))
            features, prices = features.loc[keep], prices.loc[keep]
        removed.append(len(fit_index) - len(features))
        model = provisional_model().fit(features, np.log(prices))
        predictions.iloc[valid_index] = np.exp(model.predict(X.iloc[valid_index]))
    assert predictions.notna().all()
    expensive = y.ge(y.quantile(0.95))
    scores.append([option, removed, rmse_log(y, predictions), cv_rmse(y, predictions, X),
                   rmse_log(y[expensive], predictions[expensive])])

comparison = pd.DataFrame(scores, columns=["Option", "Retraits par pli", "RMSE log · toutes",
                                          "RMSE log · ventes typiques",
                                          "RMSE log · 5 % les plus chères"])
display(comparison.round(4))

**2.6 Valeurs aberrantes · comparaison des retraits**

,Option,Retraits par pli,RMSE log · toutes,RMSE log · ventes typiques,RMSE log · 5 % les plus chères
0,A · Tout garder,"[0, 0, 0, 0, 0]",0.1325,0.1183,0.1772
1,B · Retrait ciblé,"[2, 2, 0, 2, 2]",0.1283,0.1137,0.1448
2,C · Surface > 4 000 pi²,"[3, 3, 2, 4, 4]",0.1276,0.1136,0.1474
3,D · Prix du centile supérieur,"[12, 12, 12, 12, 12]",0.1337,0.1214,0.1993


**Interprétation.** Le retrait du centile supérieur (D) prive le modèle de ventes représentatives du haut de gamme. Sur les 5 % les plus chères, l’erreur passe de 0,1772 (A) à 0,1993 (D). Une maison de test dépasse également 4 000 pi², même si son prix est inconnu. B et C sont proches sur toutes les ventes (0,1283 et 0,1276) comme sur les ventes typiques (0,1137 et 0,1136). Sur les ventes typiques, retirer les deux ventes ciblées de l’entraînement réduit nettement l’erreur : 0,1183 pour A contre 0,1137 pour B. B donne la plus faible erreur sur les maisons chères (0,1448 contre 0,1474).

**Décision.** Nous retenons B. De Cock (2011), l’auteur du jeu de données, conseille d’écarter toutes les maisons de plus de 4 000 pi² (option C), mais il précise qu’elles sont de deux types : des ventes partielles dont le prix ne reflète pas la valeur de marché, et de très grandes maisons vendues à un prix cohérent. Dans le train Kaggle, 524 et 1299 sont du premier type, et les deux maisons de NoRidge du second. B retire seulement les premières et garde les deux ventes de NoRidge, qui suivent la tendance générale. [De Cock (2011), *Journal of Statistics Education*, 19(3)](https://doi.org/10.1080/10691898.2011.11889627)

### 2.7 Récapitulatif du nettoyage

Les suppressions de colonnes et la conversion de type concernent chaque ligne. Pour les corrections de valeurs, nous comptons les différences observées entre les données brutes et la sortie de `Cleaner`. La dernière ligne compte le retrait ciblé sur l’ensemble de train à titre descriptif.

Le nettoyage conserve toutes les lignes nécessaires aux prédictions Kaggle. Les contradictions de maçonnerie laissent une information exploitable dans l’autre champ. La prochaine étape traitera les valeurs manquantes en tenant compte de cet indice.

In [13]:
# ──────────────────────────────────────────────────────────────────────
# 2.7 Récapitulatif du nettoyage
# ──────────────────────────────────────────────────────────────────────
show_title("2.7 Récapitulatif du nettoyage")
def counts(mask):
    """Compter les maisons concernées dans chacun des deux fichiers."""
    return [int(mask.loc[file].sum()) for file in ("train", "test")]

sizes = [len(train), len(test)]
retained, _ = remove_outliers(X, y)
summary = pd.DataFrame([
    ["Doublons", train.duplicated().sum(), test.duplicated().sum(), "Aucun retrait"],
    ["Id", *sizes, "Retirer la colonne"],
    ["MSSubClass", *sizes, "Convertir en variable catégorielle"],
    ["Exterior2nd", *counts(changed("Exterior2nd")), "Harmoniser trois codes"],
    ["Utilities", *sizes, "Retirer la colonne"],
    ["YearRemodAdd", *counts(changed("YearRemodAdd")), "Reprendre YearBuilt"],
    ["GarageYrBlt", *counts(changed("GarageYrBlt")), "Reprendre YearRemodAdd"],
    ["Parement absent, surface minime", *counts(placeholder), "Ramener la surface à 0"],
    ["Parement absent, surface > 1", *counts(type_changed), "Marquer le type inconnu"],
    ["Matériau connu, surface nulle", *counts(unknown_area), "Marquer la surface inconnue"],
    ["Ventes aberrantes (B)", len(X) - len(retained), 0, "Retirer de l’entraînement seulement"],
], columns=["Règle", "Maisons dans train", "Maisons dans test", "Action"])
display(summary)

**2.7 Récapitulatif du nettoyage**

,Règle,Maisons dans train,Maisons dans test,Action
0,Doublons,0,0,Aucun retrait
1,Id,1460,1459,Retirer la colonne
2,MSSubClass,1460,1459,Convertir en variable catégorielle
3,Exterior2nd,105,124,Harmoniser trois codes
4,Utilities,1460,1459,Retirer la colonne
5,YearRemodAdd,0,1,Reprendre YearBuilt
6,GarageYrBlt,0,1,Reprendre YearRemodAdd
7,"Parement absent, surface minime",2,1,Ramener la surface à 0
8,"Parement absent, surface > 1",3,1,Marquer le type inconnu
9,"Matériau connu, surface nulle",2,1,Marquer la surface inconnue


## 3. Valeurs manquantes

La section 3 du [notebook 03](03_analyse_exploratoire.ipynb) compte les valeurs absentes dans les fichiers bruts. Ici, nous examinons les caractéristiques **après nettoyage**. Un même `NA` peut avoir quatre sens.

**Quatre sens d’une valeur absente**

| Famille | Ce que signifie la valeur absente | Exemple | Traitement |
|---|---|---|---|
| 1. Équipement absent | La maison ne possède pas l’équipement | Aucun sous-sol | Catégorie `None` et quantités nulles |
| 2. Fausse absence | Une autre colonne atteste la présence | Piscine mesurée, qualité inconnue | Mode ou médiane des maisons équipées |
| 3. Valeur sans objet | La grandeur n’existe pas pour cette maison | Année d’un garage inexistant | Convention numérique fondée sur l’année de la maison |
| 4. Information inconnue | La valeur existe mais manque dans la fiche | Longueur de façade du terrain | Médiane du quartier, règle du dictionnaire ou statistique globale |

Nous passons en revue ces familles par groupe d’attributs.

### 3.1 Équipements

Le dictionnaire décrit chaque équipement (sous-sol, garage, foyer, piscine, parement, équipement divers) par une **mesure** (surface, nombre de foyers, valeur) et des **descriptions** (qualité, type, finition). La mesure indique si l’équipement existe, et donc ce que signifie un NA dans ses descriptions. `MissingValueHandler` classe chaque maison dans l’un de trois cas, traités ci-dessous :

**Trois cas pour un équipement**

| Cas | Mesure | Descriptions | Traitement |
|---|---|---|---|
| Équipement absent | Nulle ou manquante | Toutes vides | Catégorie `None` et quantités à 0 |
| Description manquante | Positive | Au moins une vide | Mode ou médiane des maisons équipées de train |
| Mesure manquante | Manquante | Au moins une renseignée | Médiane de la mesure, puis comme le cas précédent |

La sortie « 3.1 Équipements » donne les colonnes de chaque groupe, puis le nombre de maisons de chaque cas avant imputation. Elle applique ensuite `MissingValueHandler` (ajusté sur train) aux deux fichiers et compte les maisons transformées dans chaque cas. La fonction `imputation_report` liste chaque champ rempli; les cas 1 à 3 ci-dessous en détaillent le contenu. 

Les attributs `Alley` (accès par une ruelle) et `Fence` (clôture) n’ont aucune mesure : selon le dictionnaire, leur NA signifie toujours une absence. La colonne « NA sans mesure » en donne le nombre, et ces NA deviennent automatiquement `None`. 

La colonne "Complètes" désigne les équipements présents dont tous les champs sont renseignés. Les modes et les médianes sont calculés sur les maisons **équipées** du train set seulement. Le code d’absence `None` n’entre pas dans les modes.

In [14]:
# ──────────────────────────────────────────────────────────────────────
# 3.1 Équipements
# ──────────────────────────────────────────────────────────────────────
show_title("3.1 Équipements")
show_title("Variables associées à chaque équipement")
display(pd.DataFrame.from_dict(EQUIPMENT_GROUPS, orient="index").rename(columns={
    "measure": "Mesure", "descriptors": "Descriptions", "numbers": "Autres quantités",
}))
before = clean.copy()
training = before.loc["train"]
reports = {file: consistency_report(before.loc[file]) for file in ("train", "test")}
show_title("Nombre de maisons par cas, avant imputation")
display(pd.concat(reports, names=["Fichier", "Groupe"]).rename(columns=CASE_NAMES))

# Imputation : MissingValueHandler apprend sur train, puis transforme les deux fichiers.
handler = MissingValueHandler().fit(training)
after = pd.concat({file: handler.transform(before.loc[file]) for file in ("train", "test")},
                  names=before.index.names)
case_names = CASE_NAMES
report = imputation_report(before, after)
filled_fields = report.reset_index(drop=True).assign(
    Fichier=report.index.get_level_values("Fichier"),
    Id=raw.loc[report.index, "Id"].to_numpy(),
).rename(columns={"Après": "Valeur imputée"})[["Cas", "Fichier", "Id", "Groupe", "Champ", "Valeur imputée"]]


**3.1 Équipements**

**Variables associées à chaque équipement**

,Mesure,Descriptions,Autres quantités
Basement,TotalBsmtSF,"[BsmtQual, BsmtCond, BsmtExposure, BsmtFinType...","[BsmtFinSF1, BsmtFinSF2, BsmtUnfSF, BsmtFullBa..."
Garage,GarageArea,"[GarageType, GarageFinish, GarageQual, GarageC...",[GarageCars]
Fireplace,Fireplaces,[FireplaceQu],[]
Pool,PoolArea,[PoolQC],[]
MasonryVeneer,MasVnrArea,[MasVnrType],[]
MiscFeature,MiscVal,[MiscFeature],[]


**Nombre de maisons par cas, avant imputation**

Complètes  Équipement absent  Description manquante  \
Fichier Groupe                                                               
train   Basement            1421                 37                      2   
        Garage              1379                 81                      0   
        Fireplace            770                690                      0   
        Pool                   7               1453                      0   
        MasonryVeneer        586                869                      3   
        MiscFeature           52               1406                      0   
        Alley               <NA>               <NA>                   <NA>   
        Fence               <NA>               <NA>                   <NA>   
test    Basement            1410                 42                      7   
        Garage              1381                 76                      1   
        Fireplace            729                730                      0   
        Pool                   3               1453                      3   
        MasonryVeneer        564                892                      2   
        MiscFeature           50               1407                      1   
        Alley               <NA>               <NA>                   <NA>   
        Fence               <NA>               <NA>                   <NA>   

                       Mesure manquante  NA sans mesure  
Fichier Groupe                                           
train   Basement                      0            <NA>  
        Garage                        0            <NA>  
        Fireplace                     0            <NA>  
        Pool                          0            <NA>  
        MasonryVeneer                 2            <NA>  
        MiscFeature                   0            <NA>  
        Alley                      <NA>            1369  
        Fence                      <NA>            1179  
test    Basement                      0            <NA>  
        Garage                        1            <NA>  
        Fireplace                     0            <NA>  
        Pool                          0            <NA>  
        MasonryVeneer                 1            <NA>  
        MiscFeature                   0            <NA>  
        Alley                      <NA>            1352  
        Fence                      <NA>            1169

#### Cas 1 · Équipement absent

**Problème.** Une maison sans équipement a une mesure nulle et des descriptions à NA : par exemple, 690 maisons de train sans foyer ont `FireplaceQu` = NA. Normalement, une maison sans l’équipement a ses quantités à 0. Quelques maisons ont plutôt ces quantités à NA : 
- La maison de test 2121 pour `TotalBsmtSF`, `BsmtFinSF1`, `BsmtFinSF2`, `BsmtUnfSF`, `BsmtFullBath` et `BsmtHalfBath`
- La maison de test 2189 pour `BsmtFullBath` et `BsmtHalfBath`
- 23 maisons (8 dans train, 15 dans test) pour `MasVnrArea`, avec `MasVnrType` aussi à NA. 

`Alley` et `Fence` n’ont pas de mesure associée, mais le dictionnaire définit leur NA comme une absence (pas d’accès par ruelle, pas de clôture).

**Décision.** Remplacer les descriptions manquantes par la catégorie `None` et les quantités manquantes par 0. 

**Code.** `MissingValueHandler.transform`.

**Justification.** Le dictionnaire traduit ces NA par « No Basement », « No Garage », « No Fireplace », etc. Une mesure nulle ou absente, sans aucune description, ne contredit pas cette lecture. Pour un parement dont rien n’est renseigné, l’absence reste une convention : les données ne permettent pas de confirmer qu’il existe. La sortie « 3.1 Équipement absent » compte, pour chaque champ, les maisons modifiées dans chaque fichier, puis le total des maisons touchées par la règle.

In [15]:
# ──────────────────────────────────────────────────────────────────────
# 3.1 Équipement absent
# ──────────────────────────────────────────────────────────────────────
show_title("3.1 Équipement absent")
rule_doc("Équipement absent", MissingValueHandler.transform)
rule_doc("Alley et Fence", MissingValueHandler.transform)
absent = filled_fields[filled_fields["Cas"].eq(case_names["R1"])].copy()
absent["Correction"] = np.where(absent["Valeur imputée"].eq("None"), "NA → None", "NA → 0")
show_title("Champs modifiés par la règle")
display(absent.groupby(["Groupe", "Champ", "Correction", "Fichier"]).size()
        .unstack("Fichier", fill_value=0)[["train", "test"]].rename_axis(columns=None))
show_title("Maisons modifiées par la règle")
display(pd.DataFrame({
    "Maisons modifiées": absent.groupby("Fichier")["Id"].nunique(),
    "Maisons du fichier": before.groupby(level="Fichier").size(),
}).loc[["train", "test"]])

**3.1 Équipement absent**

MissingValueHandler.transform — Équipement absent (mesure nulle ou manquante, descriptions vides) : descriptions à None, quantités à 0.
MissingValueHandler.transform — Alley et Fence passent à None, car le dictionnaire code ainsi leur absence.


**Champs modifiés par la règle**

train  test
Groupe        Champ        Correction             
Alley         Alley        NA → None    1369  1352
Basement      BsmtCond     NA → None      37    42
              BsmtExposure NA → None      37    42
              BsmtFinSF1   NA → 0          0     1
              BsmtFinSF2   NA → 0          0     1
              BsmtFinType1 NA → None      37    42
              BsmtFinType2 NA → None      37    42
              BsmtFullBath NA → 0          0     2
              BsmtHalfBath NA → 0          0     2
              BsmtQual     NA → None      37    42
              BsmtUnfSF    NA → 0          0     1
              TotalBsmtSF  NA → 0          0     1
Fence         Fence        NA → None    1179  1169
Fireplace     FireplaceQu  NA → None     690   730
Garage        GarageCond   NA → None      81    76
              GarageFinish NA → None      81    76
              GarageQual   NA → None      81    76
              GarageType   NA → None      81    76
MasonryVeneer MasVnrArea   NA → 0          8    15
              MasVnrType   NA → None       8    15
MiscFeature   MiscFeature  NA → None    1406  1407
Pool          PoolQC       NA → None    1453  1453

**Maisons modifiées par la règle**

,Maisons modifiées,Maisons du fichier
Fichier,,
train,1460,1460
test,1459,1459


#### Cas 2 · Description manquante

**Problème.** L’équipement existe, puisque sa mesure est positive, mais une de ses descriptions est à NA. Normalement, un équipement présent a toutes ses descriptions renseignées. Quelques maisons ont plutôt une description à NA :
- Sous-sol (`TotalBsmtSF` > 0) : maisons de train 333 pour `BsmtFinType2` et 949 pour `BsmtExposure`; maisons de test 1488 et 2349 pour `BsmtExposure`, 2041, 2186 et 2525 pour `BsmtCond`, 2218 et 2219 pour `BsmtQual`.
- Garage (`GarageArea` > 0) : maison de test 2127 pour `GarageFinish`, `GarageQual` et `GarageCond`.
- Piscine (`PoolArea` > 0) : maisons de test 2421, 2504 et 2600 pour `PoolQC`.
- Parement (`MasVnrArea` > 0) : maisons de train 625, 1301 et 1335 et maisons de test 1670 et 2611 pour `MasVnrType`. Pour 625, 1301, 1335 et 1670, le NA vient de la section 2.5, qui a marqué le matériau inconnu.
- Équipement divers (`MiscVal` > 0) : maison de test 2550 pour `MiscFeature`.

**Décision.** Remplacer chaque description manquante par la valeur la plus fréquente (mode) parmi les maisons équipées de train.

**Code.** `MissingValueHandler.transform`.

**Justification.** Écrire `None` contredirait la mesure positive. Sans autre information, le mode des maisons équipées est la valeur la plus probable. Les maisons sans l’équipement n’entrent pas dans ce calcul; sinon, le mode serait presque toujours `None`.

In [16]:
# ──────────────────────────────────────────────────────────────────────
# 3.1 Description manquante
# ──────────────────────────────────────────────────────────────────────
show_title("3.1 Description manquante")
rule_doc("Description manquante", MissingValueHandler.transform)
show_title("Champs imputés : description manquante")
display(filled_fields[filled_fields["Cas"].eq(case_names["R2"])].drop(columns="Cas")
        .sort_values(["Fichier", "Id", "Champ"]).style.hide(axis="index"))

**3.1 Description manquante**

MissingValueHandler.transform — Description manquante (mesure positive) : mode ou médiane chez les maisons équipées de train.


**Champs imputés : description manquante**

Fichier,Id,Groupe,Champ,Valeur imputée
test,1488,Basement,BsmtExposure,No
test,1670,MasonryVeneer,MasVnrType,BrkFace
test,2041,Basement,BsmtCond,TA
test,2127,Garage,GarageCond,TA
test,2127,Garage,GarageFinish,Unf
test,2127,Garage,GarageQual,TA
test,2186,Basement,BsmtCond,TA
test,2218,Basement,BsmtQual,TA
test,2219,Basement,BsmtQual,TA
test,2349,Basement,BsmtExposure,No


#### Cas 3 · Mesure manquante

**Problème.** Une description atteste l’équipement, mais sa mesure est à NA. Normalement, un équipement décrit a une mesure renseignée. Quelques maisons ont plutôt leur mesure à NA :
- Garage (`GarageType` renseigné) : maison de test 2577 pour `GarageArea` et `GarageCars`, ainsi que `GarageFinish`, `GarageQual` et `GarageCond`.
- Parement (`MasVnrType` renseigné) : maisons de train 689 et 1242 et maison de test 2320 pour `MasVnrArea`. Ce NA vient de la section 2.5, qui a marqué la surface inconnue.

**Décision.** Remplacer la mesure et les quantités par la médiane des maisons équipées de train : 484 pi² pour `GarageArea`, 2 pour `GarageCars`, 204 pi² pour `MasVnrArea`. Compléter ensuite les descriptions comme au cas 2 : `Unf` pour `GarageFinish`, `TA` pour `GarageQual` et `GarageCond`.

**Code.** `MissingValueHandler.transform`.

**Justification.** Une mesure de 0 effacerait un équipement qui existe. La médiane donne une taille typique. Ce cas est traité avant le cas 2 : une fois sa surface remplie, le garage 2577 reçoit aussi ses descriptions.

In [17]:
# ──────────────────────────────────────────────────────────────────────
# 3.1 Mesure manquante
# ──────────────────────────────────────────────────────────────────────
show_title("3.1 Mesure manquante")
rule_doc("Mesure manquante", MissingValueHandler.transform)
show_title("Champs imputés : mesure manquante")
display(filled_fields[filled_fields["Cas"].eq(case_names["R3"])].drop(columns="Cas")
        .sort_values(["Fichier", "Id", "Champ"]).style.hide(axis="index"))

**3.1 Mesure manquante**

MissingValueHandler.transform — Mesure manquante (description renseignée) : médiane de la mesure chez les maisons équipées de train.


**Champs imputés : mesure manquante**

Fichier,Id,Groupe,Champ,Valeur imputée
test,2320,MasonryVeneer,MasVnrArea,204.000000
test,2577,Garage,GarageArea,484.000000
test,2577,Garage,GarageCars,2.000000
test,2577,Garage,GarageCond,TA
test,2577,Garage,GarageFinish,Unf
test,2577,Garage,GarageQual,TA
train,689,MasonryVeneer,MasVnrArea,204.000000
train,1242,MasonryVeneer,MasVnrArea,204.000000


### 3.2 Année du garage, façade et fonctionnalité

Trois colonnes ont des valeurs manquantes qui ne dépendent pas de la présence d’un équipement : une valeur sans objet (l’année d’un garage inexistant) et deux informations inconnues (la façade du terrain et la fonctionnalité).

#### Année du garage

**Problème.** `GarageYrBlt` donne l’année de construction du garage. Un modèle exige généralement un nombre dans chaque colonne, mais `GarageYrBlt` est à NA dans deux situations :
- **Sans garage** (`GarageArea` = 0) : L’année d’un garage inexistant n’a pas de sens. Il faut pourtant choisir une valeur qui ne fausse pas le modèle.
- **Avec garage** : Le garage existe, mais son année n’a pas été saisie.

**Décision.** Remplacer `GarageYrBlt` par l’année de construction de la maison (`YearBuilt`).

**Code.** Option `garage_year_fill="year_built"` de `MissingValueHandler`.

**Justification.** Un zéro créerait un écart artificiel d’environ deux millénaires dans un modèle linéaire. La maison et son garage ont la même année dans environ 79 % des cas (sortie « 3.2 Année du garage »), ce qui fait de `YearBuilt` la valeur la plus réaliste. La variable `HasGarage`, qui sera créée à la section 4, distinguera les maisons sans garage.

In [18]:
# ──────────────────────────────────────────────────────────────────────
# 3.2 Année du garage
# ──────────────────────────────────────────────────────────────────────
show_title("3.2 Année du garage")
rule_doc("GarageYrBlt", MissingValueHandler.transform)
missing_year = before["GarageYrBlt"].isna()
situation = pd.Series(np.where(after["GarageArea"].gt(0), "Avec garage", "Sans garage"),
                      index=before.index, name="Situation")
garage_counts = pd.crosstab(situation[missing_year],
                     situation[missing_year].index.get_level_values("Fichier"))[["train", "test"]]
garage_counts.loc["Total"] = garage_counts.sum()
show_title("Maisons avec GarageYrBlt à NA")
display(garage_counts.rename_axis(columns=None))

show_title("Liste des maisons dont GarageYrBlt a été imputée")
display(pd.DataFrame({
    "Situation": situation[missing_year].to_numpy(),
    "Fichier": missing_year[missing_year].index.get_level_values("Fichier"),
    "Id": raw.loc[missing_year, "Id"].to_numpy(),
    "GarageType": after.loc[missing_year, "GarageType"].to_numpy(),
    "GarageArea": after.loc[missing_year, "GarageArea"].to_numpy(),
    "YearBuilt": after.loc[missing_year, "YearBuilt"].to_numpy(),
    "GarageYrBlt avant": before.loc[missing_year, "GarageYrBlt"].to_numpy(),
    "GarageYrBlt après": after.loc[missing_year, "GarageYrBlt"].to_numpy(),
}).sort_values(["Situation", "Fichier", "Id"], ascending=[True, False, True])
  .set_index(["Id"]))

**3.2 Année du garage**

MissingValueHandler.transform — GarageYrBlt reprend YearBuilt, vaut 0 ou est retirée selon garage_year_fill.


**Maisons avec GarageYrBlt à NA**

,train,test
Situation,,
Avec garage,0,2
Sans garage,81,76
Total,81,78


**Liste des maisons dont GarageYrBlt a été imputée**

,Situation,Fichier,GarageType,GarageArea,YearBuilt,GarageYrBlt avant,GarageYrBlt après
Id,,,,,,,
2127,Avec garage,test,Detchd,360.0,1910,NaN,1910.0
2577,Avec garage,test,Detchd,484.0,1923,NaN,1923.0
40,Sans garage,train,None,0.0,1955,NaN,1955.0
49,Sans garage,train,None,0.0,1920,NaN,1920.0
79,Sans garage,train,None,0.0,1968,NaN,1968.0
...,...,...,...,...,...,...,...
2894,Sans garage,test,None,0.0,1916,NaN,1916.0
2910,Sans garage,test,None,0.0,1970,NaN,1970.0
2914,Sans garage,test,None,0.0,1970,NaN,1970.0


#### Façade du terrain

**Problème.** `LotFrontage` donne la longueur du terrain le long de la rue, en pieds. Normalement, chaque terrain a une façade, donc une valeur positive. Quelques maisons ont plutôt `LotFrontage` à NA. La valeur existe, mais elle n’a pas été saisie.

**Décision.** Remplacer `LotFrontage` par la médiane de la façade dans le quartier de la maison (`Neighborhood`), calculée sur train. Pour un quartier absent de train ou sans mesure, prendre la médiane globale. Toutes les maisons concernées sont dans un quartier connu de train, donc ce repli ne sert pas ici.

**Code.** Option `lot_frontage_fill="neighborhood"` de `MissingValueHandler`.

**Justification.** La façade dépend beaucoup du quartier : les médianes vont de 21 à 91 pi. La médiane du quartier est donc plus proche de la vraie valeur que la médiane globale.

In [19]:
# ──────────────────────────────────────────────────────────────────────
# 3.2 Façade du terrain
# ──────────────────────────────────────────────────────────────────────
show_title("3.2 Façade du terrain")
rule_doc("LotFrontage", MissingValueHandler.transform)
frontages = handler.frontage_by_neighborhood_
global_frontage = handler.column_fill_["LotFrontage"]
missing_frontage = before["LotFrontage"].isna()
show_title("Maisons avec LotFrontage à NA")
display(pd.DataFrame({
    "LotFrontage à NA": missing_frontage.groupby(level="Fichier").sum(),
}).loc[["train", "test"]].T)

show_title("Médiane de la façade des quartiers")
display(pd.concat([
    frontages.sort_index(key=lambda names: names.str.lower()),
    pd.Series({"Médiane globale": global_frontage}),
]).rename("Médiane (pi)").rename_axis("Quartier").to_frame())
print(f"Étendue des médianes de quartier : {frontages.min():g} à {frontages.max():g} pi.")

neighborhood = before.loc[missing_frontage, "Neighborhood"]
show_title("Liste des maisons dont LotFrontage a été imputée")
display(pd.DataFrame({
    "Neighborhood": neighborhood.to_numpy(),
    "Fichier": missing_frontage[missing_frontage].index.get_level_values("Fichier"),
    "Id": raw.loc[missing_frontage, "Id"].to_numpy(),
    "LotFrontage avant": before.loc[missing_frontage, "LotFrontage"].to_numpy(),
    "LotFrontage après": after.loc[missing_frontage, "LotFrontage"].to_numpy(),
    "Source": np.where(neighborhood.isin(frontages.dropna().index),
                       "Médiane du quartier", "Médiane globale"),
}).sort_values(["Neighborhood", "Fichier", "Id"], ascending=[True, False, True])
  .set_index(["Id"]))

**3.2 Façade du terrain**

MissingValueHandler.transform — LotFrontage reprend la médiane du quartier si lot_frontage_fill vaut "neighborhood".


**Maisons avec LotFrontage à NA**

Fichier,train,test
LotFrontage à NA,259,227


**Médiane de la façade des quartiers**

,Médiane (pi)
Quartier,
Blmngtn,43.0
Blueste,24.0
BrDale,21.0
BrkSide,52.0
ClearCr,80.0
CollgCr,70.0
Crawfor,74.0
Edwards,65.5
Gilbert,65.0


Étendue des médianes de quartier : 21 à 91 pi.


**Liste des maisons dont LotFrontage a été imputée**

,Neighborhood,Fichier,LotFrontage avant,LotFrontage après,Source
Id,,,,,
466,Blmngtn,train,NaN,43.0,Médiane du quartier
560,Blmngtn,train,NaN,43.0,Médiane du quartier
852,Blmngtn,train,NaN,43.0,Médiane du quartier
1985,Blmngtn,test,NaN,43.0,Médiane du quartier
1986,Blmngtn,test,NaN,43.0,Médiane du quartier
...,...,...,...,...,...
1254,Veenker,train,NaN,68.0,Médiane du quartier
1737,Veenker,test,NaN,68.0,Médiane du quartier
1744,Veenker,test,NaN,68.0,Médiane du quartier


#### Fonctionnalité

**Problème.** `Functional` signale les défauts de fonctionnement de la maison. Normalement, chaque maison a une valeur: par exemple, `Typ` pour un fonctionnement typique, `Min1` pour des défauts mineurs. Quelques maisons ont plutôt `Functional` à NA.

**Décision.** Remplacer `Functional` par `Typ` (fonctionnement typique).

**Code.** `MissingValueHandler.transform`.

**Justification.** Le dictionnaire indique « Assume typical unless deductions are warranted » : sans information sur un défaut, la maison est considérée comme typique. [Dictionnaire des variables](../data/data_description.txt)

In [20]:
# ──────────────────────────────────────────────────────────────────────
# 3.2 Fonctionnalité
# ──────────────────────────────────────────────────────────────────────
show_title("3.2 Fonctionnalité")
rule_doc("Functional", MissingValueHandler.transform)
functional = before["Functional"].isna()
show_title("Fonctionnalité avant et après imputation")
display(pd.DataFrame({"Id": raw.loc[functional, "Id"], "Avant": before.loc[functional, "Functional"],
                      "Après": after.loc[functional, "Functional"]}).style.hide(axis="index"))

**3.2 Fonctionnalité**

MissingValueHandler.transform — Functional vaut Typ, la valeur que le dictionnaire suppose par défaut.


**Fonctionnalité avant et après imputation**

Id,Avant,Après
2217,nan,Typ
2474,nan,Typ


#### Comparaison des choix

Nous vérifions que ces choix ne nuisent pas au modèle. Nous reprenons le modèle provisoire de la section 2.6, sur les mêmes cinq plis, avec `MissingValueHandler` à la place de l’imputation provisoire. Nous comparons trois traitements de l’année du garage (`zero`, `year_built`, retrait de la colonne avec `drop`) en gardant la façade à `neighborhood`, puis deux traitements de la façade (`neighborhood`, `global`) en gardant l’année du garage à `year_built`.

In [21]:
# ──────────────────────────────────────────────────────────────────────
# 3.2 Comparaison des choix d’imputation
# ──────────────────────────────────────────────────────────────────────
show_title("3.2 Comparaison des choix d’imputation")
def imputation_score(garage_option, frontage_option):
    """Calculer une prédiction hors pli pour chacune des ventes étiquetées."""
    predictions = pd.Series(np.nan, index=y.index)
    for fit_index, valid_index in folds.split(X):
        features, prices = remove_outliers(X.iloc[fit_index], y.iloc[fit_index])
        imputer = MissingValueHandler(garage_year_fill=garage_option, lot_frontage_fill=frontage_option)
        model = provisional_model(imputer).fit(features, np.log(prices))
        predictions.iloc[valid_index] = np.exp(model.predict(X.iloc[valid_index]))
    assert predictions.notna().all()
    return cv_rmse(y, predictions, X)

garage_scores = pd.Series({option: imputation_score(option, "neighborhood")
                           for option in ["zero", "year_built", "drop"]}, name="RMSE log")
frontage_scores = pd.Series({"neighborhood": garage_scores["year_built"],
                            "global": imputation_score("year_built", "global")}, name="RMSE log")
show_title("RMSE log selon l’option pour GarageYrBlt")
display(garage_scores.rename_axis("Option GarageYrBlt").round(4).to_frame())
show_title("RMSE log selon l’option pour LotFrontage")
display(frontage_scores.rename_axis("Option LotFrontage").round(4).to_frame())

**3.2 Comparaison des choix d’imputation**

**RMSE log selon l’option pour GarageYrBlt**

,RMSE log
Option GarageYrBlt,
zero,0.1135
year_built,0.1139
drop,0.1135


**RMSE log selon l’option pour LotFrontage**

,RMSE log
Option LotFrontage,
neighborhood,0.1139
global,0.1138


**Décision.** Retenir `year_built` pour l’année du garage et `neighborhood` pour la façade. 

Avec des scores entre 0,1135 et 0,1139, l'écart maximal de 0,0004 est trop faible pour départager les options, même si `year_built` et `neighborhood` sont très légèrement derrière.

Malgré ces écarts, nous choisissons donc les options qui ont le plus de sens immobilier.

### 3.3 Repli sur la médiane ou le mode de train

**Problème.** Après les règles des sections 3.1 et 3.2, quelques valeurs restent à NA. `MSZoning`, `Exterior1st`, `Exterior2nd`, `KitchenQual`, `SaleType` et `Electrical`.

**Décision.** Remplacer toute valeur encore à NA par la médiane (colonne numérique) ou le mode (colonne texte) de train : `RL` pour `MSZoning`, `VinylSd` pour `Exterior1st` et `Exterior2nd`, `TA` pour `KitchenQual`, `WD` pour `SaleType`, `SBrkr` pour `Electrical`.

**Code.** `MissingValueHandler.transform`, dernière étape.

**Justification.** Les statistiques viennent de train seulement, pour que test n’influence pas l’apprentissage. Elles sont calculées pour toutes les colonnes, puisqu’on ne sait pas d’avance où test aura des NA. 

In [22]:
# ──────────────────────────────────────────────────────────────────────
# 3.3 Repli sur la médiane ou le mode de train
# ──────────────────────────────────────────────────────────────────────
show_title("3.3 Repli sur la médiane ou le mode de train")
rule_doc("Repli global", MissingValueHandler.transform)
test_only = X.columns[X.notna().all() & test.isna().any()]
rows = []
for column in test_only:
    mask = test[column].isna()
    if column not in after:
        rows.append([column, mask.sum(), "Nettoyage", "Colonne retirée"])
        continue
    layer = "Repli global"
    if any(column in [g["measure"], *g["numbers"], *g["descriptors"]] for g in EQUIPMENT_GROUPS.values()):
        layer = "Équipement"
    elif column == "Functional":
        layer = "Dictionnaire"
    values = after.loc["test"].loc[mask, column].unique()
    rows.append([column, mask.sum(), layer, ", ".join(str(v) for v in values)])
show_title("Colonnes complètes dans train mais avec NA dans test")
display(pd.DataFrame(rows, columns=["Colonne", "NA dans test", "Règle appliquée", "Valeurs retenues"])
        .set_index("Colonne"))

# Colonnes sans règle propre : seul le repli les remplit.
special = {column for group in EQUIPMENT_GROUPS.values()
           for column in [group["measure"], *group["descriptors"], *group["numbers"]]}
special |= {*NO_MEASURE_ABSENCE, "GarageYrBlt", "LotFrontage", "Functional"}
fallback_columns = [column for column in before if column not in special and before[column].isna().any()]
missing_fallback = before[fallback_columns].isna()
show_title("Colonnes traitées par le repli")
display(pd.DataFrame({
    "NA dans train": missing_fallback.loc["train"].sum(),
    "NA dans test": missing_fallback.loc["test"].sum(),
    "Valeur retenue": [handler.column_fill_[column] for column in fallback_columns],
}).rename_axis("Colonne"))

show_title("Liste des maisons traitées par le repli, avant et après imputation")
fallback_rows = missing_fallback.stack()
fallback_rows = fallback_rows[fallback_rows].index
display(pd.DataFrame({
    "Colonne": fallback_rows.get_level_values(-1),
    "Fichier": fallback_rows.get_level_values("Fichier"),
    "Id": [raw.at[index[:2], "Id"] for index in fallback_rows],
    "Avant": [before.at[index[:2], index[2]] for index in fallback_rows],
    "Après": [after.at[index[:2], index[2]] for index in fallback_rows],
}).sort_values(["Colonne", "Fichier", "Id"], ascending=[True, False, True])
  .set_index(["Id"]))

**3.3 Repli sur la médiane ou le mode de train**

MissingValueHandler.transform — Repli global : médiane ou mode de train pour toute valeur encore absente.


**Colonnes complètes dans train mais avec NA dans test**

,NA dans test,Règle appliquée,Valeurs retenues
Colonne,,,
MSZoning,4,Repli global,RL
Utilities,2,Nettoyage,Colonne retirée
Exterior1st,1,Repli global,VinylSd
Exterior2nd,1,Repli global,VinylSd
BsmtFinSF1,1,Équipement,0.0
BsmtFinSF2,1,Équipement,0.0
BsmtUnfSF,1,Équipement,0.0
TotalBsmtSF,1,Équipement,0.0
BsmtFullBath,2,Équipement,0.0


**Colonnes traitées par le repli**

,NA dans train,NA dans test,Valeur retenue
Colonne,,,
MSZoning,0,4,RL
Exterior1st,0,1,VinylSd
Exterior2nd,0,1,VinylSd
Electrical,1,0,SBrkr
KitchenQual,0,1,TA
SaleType,0,1,WD


**Liste des maisons traitées par le repli, avant et après imputation**

,Colonne,Fichier,Avant,Après
Id,,,,
1380,Electrical,train,NaN,SBrkr
2152,Exterior1st,test,NaN,VinylSd
2152,Exterior2nd,test,NaN,VinylSd
1556,KitchenQual,test,NaN,TA
1916,MSZoning,test,NaN,RL
2217,MSZoning,test,NaN,RL
2251,MSZoning,test,NaN,RL
2905,MSZoning,test,NaN,RL
2490,SaleType,test,NaN,WD


## 4. Création de caractéristiques

La création **ajoute** des colonnes calculées à partir des colonnes existantes. 

Une colonne créée doit respecter cinq critères :

1. **Calculable sans `SalePrice`**, avec nos seules colonnes. Le prix au pi² créerait une fuite.
2. **Un sens pour un évaluateur immobilier**, pas une combinaison au hasard.
3. **Un apport nouveau** : un calcul que le modèle linéaire ne peut pas faire seul (produit, seuil) ou un résumé de plusieurs colonnes en une seule.
4. **Aucune répétition** d’une autre création.
5. **Un lien réel avec le prix**, meilleur que celui de ses parties.

Nous présentons les 14 colonnes créées par famille : les sommes (4.1), les âges (4.2), les indicateurs de présence (4.3) et l’interaction qualité × surface (4.4).

**Quatorze colonnes créées**

| Famille | Colonnes créées | Section |
|---|---|---|
| Sommes | `TotalSF`, `TotalBath`, `TotalPorchSF` | 4.1 |
| Âges | `AgeAtSale`, `YearsSinceRemodel` | 4.2 |
| Indicateurs de présence | `HasBasement`, `HasGarage`, `HasFireplace`, `HasPool`, `HasMasonryVeneer`, `HasMiscFeature`, `Has2ndFloor`, `IsNewHouse` | 4.3 |
| Interaction | `QualxArea` | 4.4 |

Pour chaque colonne créée, le **Spearman** mesure son lien avec `SalePrice` sur les 1 460 ventes de train. Nous le comparons à celui de la colonne d’origine la plus liée au prix parmi ses parties (critère 5).

In [23]:
# ──────────────────────────────────────────────────────────────────────
# 4 · Colonnes ajoutées
# ──────────────────────────────────────────────────────────────────────
show_title("4 · Colonnes ajoutées")
# Mêmes données qu'à la section 3 : nettoyées, puis imputées par un MissingValueHandler ajusté sur train.
creator = FeatureCreator().fit(after.loc["train"])
created = pd.concat({file: creator.transform(after.loc[file]) for file in ("train", "test")},
                    names=after.index.names)
new_columns = [column for column in created.columns if column not in after]
features = created.loc["train"]
houses = features.set_axis(pd.Index(X["Id"], name="Id")).loc[[1, 40]]
print(f"{len(new_columns)} colonnes ajoutées : {', '.join(new_columns)}")

# Colonnes d'origine auxquelles chaque colonne créée est comparée (critère 5).
PARTS = {
    "TotalSF": ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "GrLivArea"],
    "TotalBath": ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"],
    "TotalPorchSF": ["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"],
    "AgeAtSale": ["YearBuilt"], "YearsSinceRemodel": ["YearRemodAdd"],
    **{f"Has{name}": [group["measure"]] for name, group in EQUIPMENT_GROUPS.items()},
    "Has2ndFloor": ["2ndFlrSF"], "IsNewHouse": ["YearBuilt"],
    "QualxArea": ["OverallQual", "GrLivArea"],
}

def link_table(frame, names):
    """Comparer le Spearman de chaque colonne créée à celui de sa partie la plus liée au prix."""
    rows = []
    for name in names:
        part_rho = features[PARTS[name]].corrwith(y, method="spearman")
        strongest = part_rho.abs().idxmax()
        rows.append([name, frame[name].corr(y, method="spearman"), strongest, part_rho[strongest]])
    return (pd.DataFrame(rows, columns=["Colonne", "Spearman", "Partie la plus liée",
                                        "Spearman de la partie"]).set_index("Colonne").round(2))

def examples(columns):
    """Afficher les maisons 1 et 40 de train, une ligne par maison."""
    return houses[columns].assign(Fichier="train")[["Fichier", *columns]]

**4 · Colonnes ajoutées**

14 colonnes ajoutées : TotalSF, TotalBath, TotalPorchSF, AgeAtSale, YearsSinceRemodel, HasBasement, HasGarage, HasFireplace, HasPool, HasMasonryVeneer, HasMiscFeature, Has2ndFloor, IsNewHouse, QualxArea


### 4.1 Sommes

**Problème.** Plusieurs colonnes décrivent des morceaux d’une même grandeur:
- La surface est répartie entre `TotalBsmtSF`, `1stFlrSF` et `2ndFlrSF`
- Les salles de bain entre `FullBath`, `HalfBath`, `BsmtFullBath` et `BsmtHalfBath`
- Les galeries et terrasses entre `WoodDeckSF`, `OpenPorchSF`, `EnclosedPorch`, `3SsnPorch` et `ScreenPorch`. 

Un acheteur regarde pourtant la grandeur totale : la surface de la maison, le nombre de salles de bain.

**Décision.** Ajouter trois sommes :
- `TotalSF` = `TotalBsmtSF` + `1stFlrSF` + `2ndFlrSF`;
- `TotalBath` = `FullBath` + 0,5 × `HalfBath` + `BsmtFullBath` + 0,5 × `BsmtHalfBath`. Une salle d’eau (toilette et lavabo) compte pour une demi-salle de bain, comme dans l’usage immobilier nord-américain;
- `TotalPorchSF` = `WoodDeckSF` + `OpenPorchSF` + `EnclosedPorch` + `3SsnPorch` + `ScreenPorch`.

**Code.** `FeatureCreator.transform`.

**Justification.** La corrélation de `TotalSF` ( 0,82) dépasse celle `GrLivArea` (0,73), et `TotalBath` (0,70) dépasse `FullBath` (0,64). `TotalPorchSF` (0,43) reste un peu sous `OpenPorchSF` (0,48). Le critère 5 n’est donc pas totalement atteint, mais la somme résume cinq colonnes souvent nulles. 

In [24]:
# ──────────────────────────────────────────────────────────────────────
# 4.1 Sommes
# ──────────────────────────────────────────────────────────────────────
show_title("4.1 Sommes")
for name in SUM_FEATURES:
    rule_doc(name, FeatureCreator.transform)
show_title("Lien avec le prix : chaque somme et sa partie la plus liée")
display(link_table(features, SUM_FEATURES))
show_title("Exemples : maisons 1 et 40 de train")
display(examples([column for name, parts in SUM_FEATURES.items() for column in [*parts, name]]))

**4.1 Sommes**

FeatureCreator.transform — TotalSF additionne TotalBsmtSF, 1stFlrSF et 2ndFlrSF.
FeatureCreator.transform — TotalBath additionne FullBath et BsmtFullBath, plus 0,5 × HalfBath et 0,5 × BsmtHalfBath.
FeatureCreator.transform — TotalPorchSF additionne WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch et ScreenPorch.


**Lien avec le prix : chaque somme et sa partie la plus liée**

,Spearman,Partie la plus liée,Spearman de la partie
Colonne,,,
TotalSF,0.82,GrLivArea,0.73
TotalBath,0.70,FullBath,0.64
TotalPorchSF,0.43,OpenPorchSF,0.48


**Exemples : maisons 1 et 40 de train**

,Fichier,TotalBsmtSF,1stFlrSF,2ndFlrSF,TotalSF,FullBath,HalfBath,BsmtFullBath,BsmtHalfBath,TotalBath,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,TotalPorchSF
Id,,,,,,,,,,,,,,,,
1,train,856.0,856,854,2566.0,2,1,1.0,0.0,3.5,0,61,0,0,0,61
40,train,0.0,1152,0,1152.0,2,0,0.0,0.0,2.0,0,0,0,0,0,0


### 4.2 Âges

**Problème.** `YearBuilt` et `YearRemodAdd` sont des années, alors qu’un acheteur raisonne en âge : une maison construite en 2003 et vendue en 2008 a 5 ans. Trois maisons ont aussi une année postérieure à leur vente. Leur âge calculé serait négatif.

**Décision.** Ajouter `AgeAtSale` = `YrSold` − `YearBuilt` et `YearsSinceRemodel` = `YrSold` − `YearRemodAdd`, ramenés à 0 s’ils sont négatifs.

**Code.** `FeatureCreator.transform`.

**Justification.** Pour un modèle linéaire, l’âge apporte la même information que l’année : `AgeAtSale` a un Spearman de −0,65 contre 0,65 pour `YearBuilt`, et `YearsSinceRemodel` −0,58 contre 0,57 pour `YearRemodAdd`. Son intérêt est la lecture : un coefficient par année d’âge se commente directement. Une maison vendue avant la fin des travaux (âge négatif), est neuve : son âge négatif n’a pas de sens, d’où 0.

In [25]:
# ──────────────────────────────────────────────────────────────────────
# 4.2 Âges
# ──────────────────────────────────────────────────────────────────────
show_title("4.2 Âges")
for name in AGE_FEATURES:
    rule_doc(name, FeatureCreator.transform)
show_title("Lien avec le prix : chaque âge et son année d’origine")
display(link_table(features, AGE_FEATURES))

raw_ages = pd.DataFrame({name: after["YrSold"] - after[column] for name, column in AGE_FEATURES.items()})
clipped = raw_ages.lt(0).any(axis=1)
show_title("Maisons dont un âge négatif a été ramené à 0")
display(pd.DataFrame({
    "Fichier": clipped[clipped].index.get_level_values("Fichier"),
    "Id": raw.loc[clipped, "Id"].to_numpy(),
    **{column: after.loc[clipped, column].to_numpy() for column in ["YrSold", "YearBuilt", "YearRemodAdd"]},
    **{f"{name} avant": raw_ages.loc[clipped, name].to_numpy() for name in AGE_FEATURES},
    **{f"{name} après": created.loc[clipped, name].to_numpy() for name in AGE_FEATURES},
}).set_index(["Id"]))

**4.2 Âges**

FeatureCreator.transform — AgeAtSale vaut YrSold − YearBuilt, ramené à 0 s'il est négatif.
FeatureCreator.transform — YearsSinceRemodel vaut YrSold − YearRemodAdd, ramené à 0 s'il est négatif.


**Lien avec le prix : chaque âge et son année d’origine**

,Spearman,Partie la plus liée,Spearman de la partie
Colonne,,,
AgeAtSale,-0.65,YearBuilt,0.65
YearsSinceRemodel,-0.58,YearRemodAdd,0.57


**Maisons dont un âge négatif a été ramené à 0**

,Fichier,YrSold,YearBuilt,YearRemodAdd,AgeAtSale avant,YearsSinceRemodel avant,AgeAtSale après,YearsSinceRemodel après
Id,,,,,,,,
524,train,2007,2007,2008,0,-1,0,0
2296,test,2007,2007,2008,0,-1,0,0
2550,test,2007,2008,2009,-1,-2,0,0


### 4.3 Indicateurs de présence

**Problème.** Une mesure nulle signifie souvent « absent ». Par exemple, une maison sans garage a `GarageArea` = 0, mais un modèle linéaire tentera de l'interpréter de la même manière qu'une mesure de 200 ou 500 pi², c'est-à-dire une très petite surface. Il faut donc donner un sens réel au fait d'avoir ou pas l'équipement.

**Décision.** Ajouter un indicateur qui vaut 1 si la mesure est positive, sinon 0 :
- un par équipement de la section 3.1 : `HasBasement` (`TotalBsmtSF`), `HasGarage` (`GarageArea`), `HasFireplace` (`Fireplaces`), `HasPool` (`PoolArea`), `HasMasonryVeneer` (`MasVnrArea`), `HasMiscFeature` (`MiscVal`);
- `Has2ndFloor` (`2ndFlrSF`), car un étage change le type de maison;
- `IsNewHouse`, qui vaut 1 si `YrSold` ≤ `YearBuilt` + 1 : maison vendue l’année de sa construction ou la suivante, y compris sur plan.

**Code.** `FeatureCreator.transform`.

**Justification.** Un indicateur sépare « absent » de « petit », ce qu’une droite sur la mesure ne fait pas. Son Spearman est souvent plus faible que celui de la mesure, car il perd la grandeur. `HasGarage` est indispensable : c’est le seul signal d’absence de garage après l’imputation de `GarageYrBlt`. Le tableau d’exemples le montre : la maison 1 (avec garage) et la maison 40 (sans garage) ont toutes deux `GarageYrBlt` = `YearBuilt`, et seul `HasGarage` les distingue. `HasPool` (Spearman 0,06; 7 piscines dans train) et `HasMiscFeature` (−0,06) n’ont presque aucun lien avec le prix. 

In [26]:
# ──────────────────────────────────────────────────────────────────────
# 4.3 Indicateurs de présence
# ──────────────────────────────────────────────────────────────────────
show_title("4.3 Indicateurs de présence")
rule_doc("Indicateurs", FeatureCreator.transform)
rule_doc("IsNewHouse", FeatureCreator.transform)
indicators = [column for column in new_columns if column.startswith("Has")] + ["IsNewHouse"]
show_title("Maisons dont l’indicateur vaut 1, et lien avec le prix")
display(created[indicators].groupby(level="Fichier").sum().T[["train", "test"]]
        .rename(columns={"train": "Maisons train", "test": "Maisons test"})
        .join(link_table(features, indicators)))
show_title("Exemples : maison 1 avec garage, maison 40 sans sous-sol ni garage")
display(examples(["TotalBsmtSF", "HasBasement", "GarageArea", "HasGarage", "GarageYrBlt", "YearBuilt"]))

**4.3 Indicateurs de présence**

FeatureCreator.transform — Indicateurs Has… : 1 si la mesure de l'équipement (ou 2ndFlrSF) est positive, sinon 0.
FeatureCreator.transform — IsNewHouse vaut 1 si YrSold ≤ YearBuilt + 1, sinon 0.


**Maisons dont l’indicateur vaut 1, et lien avec le prix**

,Maisons train,Maisons test,Spearman,Partie la plus liée,Spearman de la partie
HasBasement,1423,1417,0.20,TotalBsmtSF,0.60
HasGarage,1379,1383,0.30,GarageArea,0.65
HasFireplace,770,729,0.53,Fireplaces,0.52
HasPool,7,6,0.06,PoolArea,0.06
HasMasonryVeneer,591,567,0.39,MasVnrArea,0.42
HasMiscFeature,52,51,-0.06,MiscVal,-0.06
Has2ndFloor,631,620,0.16,2ndFlrSF,0.29
IsNewHouse,164,171,0.35,YearBuilt,0.65


**Exemples : maison 1 avec garage, maison 40 sans sous-sol ni garage**

,Fichier,TotalBsmtSF,HasBasement,GarageArea,HasGarage,GarageYrBlt,YearBuilt
Id,,,,,,,
1,train,856.0,1,548.0,1,2003.0,2003
40,train,0.0,0,0.0,0,1955.0,1955


### 4.4 Interaction qualité × surface

**Problème.** Un modèle linéaire additionne les effets : un pi² supplémentaire ajoute le même montant, que la maison soit de qualité ordinaire ou excellente. Or un pi² de grande qualité vaut plus qu’un pi² ordinaire.

**Décision.** Ajouter `QualxArea` = `OverallQual` × `GrLivArea`.

**Code.** `FeatureCreator.transform`.

**Justification.** `QualxArea` (Spearman 0,85) dépasse `OverallQual` (0,81) et `GrLivArea` (0,73). C’est probablement le seul calcul de la liste qu’un modèle linéaire ne peut pas construire lui-même à partir des colonnes d’origine.

In [27]:
# ──────────────────────────────────────────────────────────────────────
# 4.4 Interaction qualité × surface
# ──────────────────────────────────────────────────────────────────────
show_title("4.4 Interaction qualité × surface")
rule_doc("QualxArea", FeatureCreator.transform)
show_title("Lien avec le prix : QualxArea et ses parties")
display(link_table(features, ["QualxArea"]))
show_title("Exemples : maisons 1 et 40 de train")
display(examples(["OverallQual", "GrLivArea", "QualxArea"]))

**4.4 Interaction qualité × surface**

FeatureCreator.transform — QualxArea vaut OverallQual × GrLivArea.


**Lien avec le prix : QualxArea et ses parties**

,Spearman,Partie la plus liée,Spearman de la partie
Colonne,,,
QualxArea,0.85,OverallQual,0.81


**Exemples : maisons 1 et 40 de train**

,Fichier,OverallQual,GrLivArea,QualxArea
Id,,,,
1,train,7,1710,11970
40,train,4,1152,4608


## 5. Encodage des variables catégorielles

Les modèles ne calculent qu’avec des nombres. Après la section 4, 43 colonnes contiennent encore du texte, comme `KitchenQual` ou `Neighborhood`. Nous utilisons deux encodages :

- **Ordinal** : chaque niveau reçoit son rang dans l’échelle du dictionnaire (Po = 1 < Fa = 2 < TA = 3 < Gd = 4 < Ex = 5). Il convient seulement aux variables dont les niveaux ont un ordre réel de qualité ou de degré.
- **One-hot** : une colonne 0/1 par catégorie. Aucune catégorie ne reçoit un nombre plus grand qu’une autre : c’est l’encodage des variables sans ordre.

Numéroter les catégories dans l’ordre alphabétique serait une erreur : pour `KitchenQual`, on aurait Ex = 0, Fa = 1, Gd = 2 et TA = 3, et une cuisine moyenne (TA) « vaudrait » plus qu’une cuisine excellente (Ex).

La section 5.1 classe les 43 variables. Les sections 5.2 et 5.3 présentent les deux encodages, la section 5.4 regroupe les catégories rares et la section 5.5 résume l’encodage.

### 5.1 Variables à encoder

**Règle.** Une variable reçoit l’encodage ordinal seulement si le [dictionnaire des variables](../data/data_description.txt) décrit une échelle où monter d’un niveau signifie « mieux » ou « plus ». Les autres reçoivent le one-hot. Pour les variables ordinales, `None` (équipement absent) vaut 0, sous le niveau le plus bas.

**Variables texte et leur encodage**

| Variables | Encodage | Justification |
|---|---|---|
| `ExterQual`, `ExterCond`, `BsmtCond`, `HeatingQC`, `KitchenQual`, `FireplaceQu`, `GarageQual`, `GarageCond` | Ordinal | Échelle Ex > Gd > TA > Fa > Po du dictionnaire |
| `BsmtQual` | Ordinal | Hauteur du sous-sol, même échelle Ex > Gd > TA > Fa > Po |
| `PoolQC` | Ordinal | Échelle Ex > Gd > TA > Fa |
| `BsmtExposure` | Ordinal | Exposition Gd > Av > Mn > No |
| `BsmtFinType1`, `BsmtFinType2` | Ordinal | Finition GLQ > ALQ > BLQ > Rec > LwQ > Unf |
| `GarageFinish` | Ordinal | Finition Fin > RFn > Unf |
| `Functional` | Ordinal | Déductions croissantes, de Typ à Sal |
| `LotShape`, `LandSlope`, `PavedDrive`, `CentralAir`, `Street`, `Alley` | One-hot | Cas discutables : un degré d’irrégularité, de pente ou de pavage, pas clairement une valeur |
| `Fence` | One-hot | Mélange deux dimensions, l’intimité (GdPrv, MnPrv) et le bois (GdWo, MnWw) |
| `MSSubClass` | One-hot | Code de type de logement : ses nombres ne mesurent rien (section 2.1) |
| `MSZoning`, `Neighborhood`, `LandContour`, `LotConfig`, `Condition1`, `Condition2` | One-hot | Zones, lieux, relief, position du terrain, proximités : aucun ordre |
| `BldgType`, `HouseStyle`, `RoofStyle`, `RoofMatl`, `Exterior1st`, `Exterior2nd`, `MasVnrType`, `Foundation`, `Heating`, `Electrical`, `GarageType`, `MiscFeature` | One-hot | Types, styles et matériaux : aucun ordre |
| `SaleType`, `SaleCondition` | One-hot | Modes et conditions de vente : aucun ordre |

**Cas discutables.** `LotShape`, `LandSlope`, `PavedDrive`, `CentralAir`, `Street` et `Alley` ont un ordre apparent, mais rien n’indique qu’un niveau vaut plus qu’un autre : un terrain irrégulier est souvent un grand terrain de coin. Le one-hot laisse le modèle estimer chaque effet sans imposer de direction.

**Pourquoi l’ordinal pour les échelles, plutôt que le one-hot.** Une échelle reste une seule colonne. La sélection (section 7) garde ou rejette `KitchenQual` en entier, plutôt que des morceaux comme `KitchenQual_Ex`, et l’ACP ne reçoit pas des dizaines de colonnes 0/1 presque vides.

In [28]:
# ──────────────────────────────────────────────────────────────────────
# 5.1 Variables à encoder
# ──────────────────────────────────────────────────────────────────────
show_title("5.1 Variables à encoder")
ordered = QualityOrdinalEncoder().fit_transform(features)
nominal = nominal_columns(ordered)
show_title("Nombre de variables par encodage")
display(encoding_summary(features))

**5.1 Variables à encoder**

**Nombre de variables par encodage**

,Encodage,Variables
Groupe,,
Échelle de qualité,Ordinal,10
Autre échelle ordonnée,Ordinal,5
Nominale,One-hot,28


### 5.2 Encodage ordinal

`QualityOrdinalEncoder` remplace chaque niveau par son rang. Le tableau « Échelles ordinales » donne les cinq échelles utilisées. L’absence (`None`) vaut 0, sous le niveau le plus bas.

**Justification.** Le prix médian monte à chaque niveau de `KitchenQual` (tableau « Prix médian par niveau de KitchenQual ») : l’ordre du dictionnaire est aussi celui du prix. Les écarts ne sont toutefois pas égaux : 64 400 \$ de TA à Gd, puis 115 350 \$ de Gd à Ex. Un seul coefficient par échelle ne reproduit pas ces écarts croissants : c’est la limite de l’ordinal.

Le tableau « Avant et après l’encodage ordinal » montre trois maisons de train. Les équipements absents de la maison 40 (sous-sol, garage, foyer, piscine) valent 0. La maison 198, première maison de train avec une piscine, a une piscine et un foyer excellents (Ex), qui valent 5.

In [29]:
# ──────────────────────────────────────────────────────────────────────
# 5.2 Encodage ordinal
# ──────────────────────────────────────────────────────────────────────
show_title("5.2 Encodage ordinal")
scales = {}
for column, scale in ORDINAL_COLUMNS.items():
    scales.setdefault(id(scale), (scale, []))[1].append(column)
show_title("Échelles ordinales")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([[", ".join(columns), " < ".join(f"{code} = {rank}" for code, rank in scale.items())]
                          for scale, columns in scales.values()],
                         columns=["Colonnes", "Correspondance code → nombre"]))

show_title("Prix médian par niveau de KitchenQual")
levels = (pd.DataFrame({"KitchenQual": features["KitchenQual"], "Prix": y})
          .groupby("KitchenQual")["Prix"].agg(Maisons="size", Médiane="median"))
levels["Rang"] = levels.index.map(ORDINAL_COLUMNS["KitchenQual"])
levels = levels.sort_values("Rang")[["Rang", "Maisons", "Médiane"]]
levels["Écart avec le niveau précédent"] = levels["Médiane"].diff()
display(levels)

show_title("Avant et après l’encodage ordinal")
ordinal_ids = [1, 40, int(X.loc[X["PoolArea"].gt(0), "Id"].iat[0])]
shown = ["KitchenQual", "ExterQual", "BsmtQual", "BsmtExposure", "BsmtFinType1", "GarageFinish",
         "FireplaceQu", "PoolQC", "Functional"]
ids = pd.Index(X["Id"], name="Id")
by_id = {"Avant": features.set_axis(ids), "Après": ordered.set_axis(ids)}
display(pd.concat([frame.loc[[house], shown].reset_index().assign(Fichier="train", État=state)
                   for house in ordinal_ids for state, frame in by_id.items()], ignore_index=True)
        [["Fichier", "Id", "État", *shown]])

**5.2 Encodage ordinal**

**Échelles ordinales**

,Colonnes,Correspondance code → nombre
0,"ExterQual, ExterCond, BsmtQual, BsmtCond, HeatingQC, KitchenQual, FireplaceQu, GarageQual, GarageCond, PoolQC",None = 0 < Po = 1 < Fa = 2 < TA = 3 < Gd = 4 < Ex = 5
1,BsmtExposure,None = 0 < No = 1 < Mn = 2 < Av = 3 < Gd = 4
2,"BsmtFinType1, BsmtFinType2",None = 0 < Unf = 1 < LwQ = 2 < Rec = 3 < BLQ = 4 < ALQ = 5 < GLQ = 6
3,GarageFinish,None = 0 < Unf = 1 < RFn = 2 < Fin = 3
4,Functional,Sal = 0 < Sev = 1 < Maj2 = 2 < Maj1 = 3 < Mod = 4 < Min2 = 5 < Min1 = 6 < Typ = 7


**Prix médian par niveau de KitchenQual**

,Rang,Maisons,Médiane,Écart avec le niveau précédent
KitchenQual,,,,
Fa,2,39,115000.0,NaN
TA,3,735,137000.0,22000.0
Gd,4,586,201400.0,64400.0
Ex,5,100,316750.0,115350.0


**Avant et après l’encodage ordinal**

,Fichier,Id,État,KitchenQual,ExterQual,BsmtQual,BsmtExposure,BsmtFinType1,GarageFinish,FireplaceQu,PoolQC,Functional
0,train,1,Avant,Gd,Gd,Gd,No,GLQ,RFn,None,None,Typ
1,train,1,Après,4,4,4,1,6,2,0,0,7
2,train,40,Avant,Fa,TA,None,None,None,None,None,None,Typ
3,train,40,Après,2,3,0,0,0,0,0,0,7
4,train,198,Avant,Gd,Gd,TA,No,GLQ,Unf,Ex,Ex,Typ
5,train,198,Après,4,4,3,1,6,1,5,5,7


### 5.3 Encodage one-hot

`make_onehot_encoder` crée une colonne 0/1 par catégorie, et chaque maison a un seul 1 par variable. Le tableau montre `Foundation` pour trois maisons.

In [30]:
# ──────────────────────────────────────────────────────────────────────
# 5.3 Encodage one-hot
# ──────────────────────────────────────────────────────────────────────
show_title("5.3 Encodage one-hot")
foundation_ids = [int(X.loc[features["Foundation"].eq(kind), "Id"].iat[0])
                  for kind in ["PConc", "CBlock", "BrkTil"]]
foundation_encoder = make_onehot_encoder(min_count=None).set_output(transform="pandas")
foundation_encoder.fit(features[["Foundation"]])
foundations = features.set_axis(X["Id"]).loc[foundation_ids, ["Foundation"]]
show_title("Foundation avant et après le one-hot")
display(pd.concat([foundations, foundation_encoder.transform(foundations).astype(int)], axis=1))

def make_encoder(min_count=RARE_CATEGORY_MIN_COUNT):
    """Ordinal pour les 15 échelles, puis one-hot des nominales; le reste passe tel quel."""
    columns = ColumnTransformer([("nominal", make_onehot_encoder(min_count), nominal_columns)],
                                remainder="passthrough", verbose_feature_names_out=False)
    return make_pipeline(QualityOrdinalEncoder(), columns.set_output(transform="pandas"))

encoder = make_encoder().fit(features)
onehot = encoder[-1].named_transformers_["nominal"]
onehot_columns = len(onehot.get_feature_names_out())
print(f"{len(nominal)} variables nominales → {onehot_columns} colonnes one-hot "
      f"(seuil de {RARE_CATEGORY_MIN_COUNT} maisons).")
print(f"Tableau final : {encoder.transform(features).shape[1]} colonnes, toutes numériques.")

**5.3 Encodage one-hot**

**Foundation avant et après le one-hot**

,Foundation,Foundation_BrkTil,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Foundation_Stone,Foundation_Wood
Id,,,,,,,
1,PConc,0,0,1,0,0,0
2,CBlock,0,1,0,0,0,0
4,BrkTil,1,0,0,0,0,0


28 variables nominales → 163 colonnes one-hot (seuil de 10 maisons).
Tableau final : 227 colonnes, toutes numériques.


### 5.4 Modalités rares

**Problème.** Une catégorie de 1 ou 2 maisons crée une colonne presque vide : son coefficient repose sur une ou deux ventes et n’est pas fiable. Par exemple, `RoofMatl` compte huit matériaux, dont six concernent moins de 10 maisons de train. Une catégorie peut aussi apparaître dans test sans exister dans train.

**Décision.** Regrouper les catégories de moins de 10 maisons de train (`RARE_CATEGORY_MIN_COUNT`) dans une colonne « autre ». Une catégorie absente de train rejoint aussi cette colonne.

**Code.** `make_onehot_encoder` (paramètre `min_count`), ajusté sur train seulement.

**Justification.** Le tableau « Seuil de regroupement » compare cinq seuils en validation croisée à 5 plis, avec un modèle provisoire. Jusqu’à 10 maisons, la RMSE ne bouge pas (0,1138 sans regroupement comme à 10), alors que 36 colonnes disparaissent (199 → 163). Au-delà, l’erreur augmente (0,1152 à 20, 0,1179 à 50) : on fusionne alors des catégories utiles, comme des quartiers entiers. Les variables quasi constantes de la section 2.3 (`Street`, `Condition2`, `RoofMatl`, `Heating`, `MiscFeature`) gardent ainsi leur modalité principale, et leurs exceptions partagent une colonne.

In [31]:
# ──────────────────────────────────────────────────────────────────────
# 5.4 Modalités rares
# ──────────────────────────────────────────────────────────────────────
show_title("5.4 Modalités rares")
# Les sommes et QualxArea sont des surfaces ou un produit de surface : très asymétriques.
created_areas = [*area_columns, "TotalSF", "TotalPorchSF", "QualxArea"]

def present_areas(frame):
    """Garder les surfaces présentes dans le tableau reçu."""
    return [column for column in created_areas if column in frame]

def other_created_numeric(frame):
    """Les autres grandeurs, dont les indicateurs, restent sur leur échelle."""
    return frame.select_dtypes(include="number").columns.difference(created_areas).tolist()

def cv_score(build):
    """RMSE log des prédictions hors pli, avec retrait ciblé dans l’entraînement de chaque pli."""
    predictions = pd.Series(np.nan, index=y.index)
    for fit_index, valid_index in folds.split(X):
        fit_features, fit_prices = remove_outliers(X.iloc[fit_index], y.iloc[fit_index])
        model = build().fit(fit_features, np.log(fit_prices))
        predictions.iloc[valid_index] = np.exp(model.predict(X.iloc[valid_index]))
    assert predictions.notna().all()
    return cv_rmse(y, predictions, X)

def encoding_model(min_count=RARE_CATEGORY_MIN_COUNT):
    """Modèle provisoire (log1p des surfaces, standardisation, RidgeCV) avec l’encodage retenu et le seuil min_count."""
    columns = ColumnTransformer([
        ("areas", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), present_areas),
        ("numeric", StandardScaler(), other_created_numeric),
        ("text", make_onehot_encoder(min_count), nominal_columns),
    ])
    return make_pipeline(Cleaner(), MissingValueHandler(), FeatureCreator(), QualityOrdinalEncoder(),
                         columns, RidgeCV(alphas=np.logspace(-1, 3, 30)))

nominal_train = ordered[nominal]
rare_scores = pd.DataFrame(
    [[str(t), len(make_onehot_encoder(t).fit(nominal_train).get_feature_names_out()),
      cv_score(lambda: encoding_model(t))] for t in [None, 5, 10, 20, 50]],
    columns=["Seuil (maisons)", "Colonnes one-hot", "RMSE log"]).set_index("Seuil (maisons)")
show_title("Seuil de regroupement")
display(rare_scores.round(4))

grouped = []
for column, rare in zip(onehot.feature_names_in_, onehot.infrequent_categories_):
    if rare is not None:
        category_counts = nominal_train[column].value_counts()
        grouped.append([column, len(rare), ", ".join(f"{code} ({category_counts[code]})" for code in rare)])
grouped = pd.DataFrame(grouped, columns=["Variable", "Catégories regroupées", "Détail (maisons)"])
show_title(f"Catégories regroupées dans « autre » ({len(grouped)} variables sur {len(nominal)})")
with pd.option_context("display.max_colwidth", None):
    display(grouped.set_index("Variable"))


**5.4 Modalités rares**

**Seuil de regroupement**

,Colonnes one-hot,RMSE log
Seuil (maisons),,
None,199,0.1138
5,178,0.1139
10,163,0.1138
20,144,0.1152
50,117,0.1179


**Catégories regroupées dans « autre » (18 variables sur 28)**

,Catégories regroupées,Détail (maisons)
Variable,,
MSSubClass,1,40 (4)
Street,1,Grvl (6)
LotConfig,1,FR3 (4)
Neighborhood,2,"Blueste (2), NPkVill (9)"
Condition1,3,"PosA (8), RRNe (2), RRNn (5)"
Condition2,7,"Artery (2), Feedr (6), PosA (1), PosN (2), RRAe (1), RRAn (1), RRNn (2)"
HouseStyle,1,2.5Fin (8)
RoofStyle,2,"Mansard (7), Shed (2)"
RoofMatl,6,"ClyTile (1), Membran (1), Metal (1), Roll (1), WdShake (5), WdShngl (6)"


### 5.5 Récapitulatif

Les 15 échelles gardent leur ordre en une colonne chacune. Les 28 variables nominales deviennent 163 colonnes 0/1, sans ordre imposé.

**Encodage des 43 variables texte**

| Groupe | Variables | Encodage | Colonnes produites | Justification |
|---|---|---|---|---|
| Échelles de qualité | 10 | Ordinal | 10 | Ordre Ex > Gd > TA > Fa > Po du dictionnaire; une colonne par échelle |
| Autres échelles ordonnées | 5 | Ordinal | 5 | Degré réel : exposition, finition, fonctionnalité |
| Nominales | 28 | One-hot, catégories de moins de 10 maisons regroupées | 163 | Aucun ordre; aucune catégorie ne vaut plus qu’une autre |

## 6. Logarithme, normalisation et standardisation

Après l’encodage, 64 colonnes sont numériques : les grandeurs (surfaces, comptages, années), les 15 colonnes ordinales et les indicateurs 0/1. Les 163 colonnes one-hot restent 0/1. Trois transformations peuvent s’appliquer aux colonnes numériques :

- **Logarithme** : `log1p(x)` = ln(1 + x), qui accepte les zéros. Il change la **forme** d’une distribution à longue queue en rapprochant les très grandes valeurs du reste.
- **Normalisation min-max** : (x − min) / (max − min) ramène chaque colonne entre 0 et 1.
- **Standardisation** : (x − moyenne) / écart type ramène chaque colonne à une moyenne de 0 et un écart type de 1.

La normalisation et la standardisation changent l’**échelle**, pas la forme. Nous appliquons donc d’abord le log, puis la mise à l’échelle.

La section 6.1 choisit les colonnes passées au log, la section 6.2 la mise à l’échelle, la section 6.3 précise les modèles qui en ont besoin et la section 6.4 résume.

### 6.1 Logarithme

**Problème.** Plusieurs surfaces ont une longue queue à droite : quelques très grandes maisons s’éloignent du reste. Une droite ajustée sur ces valeurs est tirée par ces quelques maisons. D’autres colonnes sont asymétriques pour une autre raison : elles valent surtout 0. `2ndFlrSF` vaut 0 pour plus de la moitié des maisons, qui n’ont pas d’étage.

**Décision.** Appliquer `log1p` à une colonne si son asymétrie dépasse 0,75 **et** si moins de 50 % des maisons valent 0. Les années, les mois, les scores, `KitchenAbvGr`, les échelles ordinales et les indicateurs 0/1 sont exclus d’office : ce ne sont pas des grandeurs à longue queue.

**Code.** `SkewCorrector`, ajusté sur train.

**Justification.**
- **Le log corrige la longue queue** Pour `LotArea`, `GrLivArea` et `1stFlrSF`, l’asymétrie tombe près de 0. Trois colonnes, `BsmtUnfSF`, `TotalPorchSF` et `LotFrontage`, deviennent asymétriques vers la gauche : le log étire leurs quelques petites valeurs. .
- **Une colonne surtout nulle n’a pas de longue queue, mais une masse de zéros.** Le log rapprocherait un petit étage d’un grand sans rapprocher les zéros du reste. La présence d’un étage est déjà dans `Has2ndFloor` (section 4.3).
- **La validation croisée confirme la règle** : 0,1146 sans log, 0,1132 avec le log de toutes les colonnes asymétriques, 0,1123 en excluant les colonnes surtout nulles.

In [32]:
# ──────────────────────────────────────────────────────────────────────
# 6.1 Logarithme
# ──────────────────────────────────────────────────────────────────────
show_title("6.1 Logarithme")
# Le pipeline s'ajuste sur l'entraînement sans les deux ventes aberrantes : nous faisons de même.
fit_rows, fit_prices = remove_outliers(ordered, y)
skew = SkewCorrector().fit(fit_rows)
rule_doc("log1p", SkewCorrector.fit)
rule_doc("Exclues", SkewCorrector.fit)

report = skew.skew_report_.set_index("Colonne")
report["Asymétrie après log1p"] = skew.transform(fit_rows)[report.index].skew()
show_title(f"Colonnes passées au log ({len(skew.log_columns_)})")
display(report.loc[skew.log_columns_, ["Asymétrie", "Part de zéros", "Asymétrie après log1p"]]
        .sort_values("Asymétrie", ascending=False).round(2))
show_title("Colonnes asymétriques laissées telles quelles, car surtout nulles")
display(report.loc[report["Raison"].eq("Surtout des zéros"), ["Asymétrie", "Part de zéros"]]
        .sort_values("Part de zéros").round(2))

def scaling_model(kind="standard", **skew_options):
    """Prétraitement des sections 2 à 5, puis log, mise à l’échelle et RidgeCV."""
    columns = ColumnTransformer([
        ("numeric", make_scaler(kind), make_column_selector(dtype_include="number")),
        ("nominal", make_onehot_encoder(), nominal_columns),
    ])
    return make_pipeline(Cleaner(), MissingValueHandler(), FeatureCreator(), QualityOrdinalEncoder(),
                         SkewCorrector(**skew_options), columns, RidgeCV(alphas=np.logspace(-1, 3, 30)))

rules = {
    "Aucun log": {"threshold": None},
    "Seuil 0,75, toutes les colonnes": {"max_zero_share": 1.0},
    "Seuil 0,75 sans colonnes surtout nulles (choix)": {},
}
show_title("RMSE log selon la règle du log")
display(pd.Series({name: cv_score(lambda: scaling_model(**options)) for name, options in rules.items()},
                  name="RMSE log").round(4).to_frame())

**6.1 Logarithme**

SkewCorrector.fit — log1p pour une colonne dont l'asymétrie dépasse threshold (0,75) et dont moins de max_zero_share (50 %) des valeurs valent 0.
SkewCorrector.fit — Exclues d'office : années, mois, scores, KitchenAbvGr, échelles ordinales (exclude) et indicateurs 0/1.


**Colonnes passées au log (10)**

,Asymétrie,Part de zéros,Asymétrie après log1p
Colonne,,,
LotArea,12.57,0.00,-0.18
OpenPorchSF,2.34,0.45,-0.02
LotFrontage,1.55,0.00,-1.00
QualxArea,1.49,0.00,-0.36
TotalPorchSF,1.10,0.17,-1.31
GrLivArea,1.01,0.00,-0.07
BsmtUnfSF,0.92,0.08,-2.18
1stFlrSF,0.89,0.00,0.01
TotalSF,0.80,0.00,-0.47


**Colonnes asymétriques laissées telles quelles, car surtout nulles**

,Asymétrie,Part de zéros
Colonne,,
WoodDeckSF,1.55,0.52
2ndFlrSF,0.81,0.57
MasVnrArea,2.69,0.60
EnclosedPorch,3.09,0.86
BsmtFinSF2,4.25,0.89
ScreenPorch,4.12,0.92
BsmtHalfBath,4.10,0.94
MiscVal,24.46,0.96
LowQualFinSF,9.00,0.98


**RMSE log selon la règle du log**

,RMSE log
Aucun log,0.1146
"Seuil 0,75, toutes les colonnes",0.1132
"Seuil 0,75 sans colonnes surtout nulles (choix)",0.1123


### 6.2 Normalisation et standardisation

**Problème.** Les colonnes numériques ont des échelles très différentes : `LotArea` se mesure en pi², `OverallQual` va de 1 à 10. Le problème dépend du modèle :
- **Arbres** (XGBoost, Gradient Boosting, forêt aléatoire), les modèles que l’étape 5 testera en priorité : aucun problème. Un arbre coupe une colonne à un seuil, et changer l’unité ne change pas l’ordre des maisons, donc pas les coupures possibles.
- **Modèles linéaires pénalisés** (Ridge, Lasso, ElasticNet) : la pénalité porte sur la taille des coefficients, qui dépend de l’unité. Le coefficient d’une colonne en pi² est minuscule et presque pas pénalisé, celui d’une note est gros et fortement pénalisé.
- **ACP** (section 7.3) : elle cherche les directions de plus grande variance. Sans mise à l’échelle, `LotArea` dominerait la première composante.

**Décision.**
- Pour un modèle à base d’arbres : aucune mise à l’échelle, avec `make_base(log=False, scaler="none")` (section 6.3).
- Pour un modèle linéaire ou l’ACP : standardiser les 64 colonnes numériques, y compris les ordinales et les indicateurs 0/1. C’est le réglage par défaut de `make_base()`, utilisé pour les comparaisons de ce notebook, faites avec Ridge.
- Dans les deux cas, les 163 colonnes one-hot restent 0/1. La normalisation min-max n’est pas retenue.

**Code.** `make_scaler()`, ajusté sur train.

**Justification.**
- **Arbres.** La mise à l’échelle ne leur apporte rien. La retirer garde des valeurs lisibles (pi², notes) et une étape de moins à ajuster.
- **Modèles linéaires et ACP.** Après standardisation, chaque coefficient mesure l’effet d’un écart type : la pénalité devient équitable entre les colonnes. Pour l’ACP, chaque colonne part avec le même poids (variance 1).
- **Pas de min-max.** Il dépend d’une seule valeur extrême, le maximum, qui comprime toutes les autres maisons près de 0. Avec Ridge, la validation croisée (tableau « RMSE log selon la mise à l’échelle ») donne 0,1134 avec le min-max, contre 0,1123 avec la standardisation, autant que sans mise à l’échelle.
- **One-hot.** Ces colonnes partagent déjà la même échelle 0/1 et restent lisibles comme « présente ou non ».

In [33]:
# ──────────────────────────────────────────────────────────────────────
# 6.2 Normalisation et standardisation
# ──────────────────────────────────────────────────────────────────────
show_title("6.2 Normalisation et standardisation")
show_title("Variance de trois colonnes avant la mise à l’échelle")
display(fit_rows[["LotArea", "GrLivArea", "OverallQual"]].var().rename("Variance").to_frame().round(1))

scalers = {"Aucune mise à l’échelle": "none", "Min-max": "minmax", "Standardisation (choix)": "standard"}
show_title("RMSE log selon la mise à l’échelle")
display(pd.Series({name: cv_score(lambda: scaling_model(kind)) for name, kind in scalers.items()},
                  name="RMSE log").round(4).to_frame())

numeric_columns = fit_rows.select_dtypes(include="number").columns
binary = [column for column in numeric_columns if fit_rows[column].nunique() <= 2]
show_title("Colonnes standardisées")
display(pd.DataFrame([
    ["Grandeurs et comptages", len(numeric_columns) - len(ORDINAL_COLUMNS) - len(binary), "Oui"],
    ["Ordinales (section 5.2)", len(ORDINAL_COLUMNS), "Oui"],
    ["Indicateurs 0/1", len(binary), "Oui"],
    ["One-hot (section 5.3)", onehot_columns, "Non, restent 0/1"],
], columns=["Colonnes", "Nombre", "Standardisées"]).set_index("Colonnes"))

**6.2 Normalisation et standardisation**

**Variance de trois colonnes avant la mise à l’échelle**

,Variance
LotArea,97203788.3
GrLivArea,257940.6
OverallQual,1.9


**RMSE log selon la mise à l’échelle**

,RMSE log
Aucune mise à l’échelle,0.1123
Min-max,0.1134
Standardisation (choix),0.1123


**Colonnes standardisées**

,Nombre,Standardisées
Colonnes,,
Grandeurs et comptages,41,Oui
Ordinales (section 5.2),15,Oui
Indicateurs 0/1,8,Oui
One-hot (section 5.3),163,"Non, restent 0/1"


### 6.3 Selon le modèle

Le log et la mise à l’échelle servent les modèles qui combinent les colonnes par des coefficients pénalisés ou des distances : Ridge, Lasso, ElasticNet, Kernel Ridge, les k plus proches voisins, ainsi que l’ACP. Les modèles à base d’arbres (forêt aléatoire, Gradient Boosting, XGBoost) n’en ont pas besoin. Un arbre coupe une colonne à un seuil, par exemple `GrLivArea` < 1 500. Le log et la mise à l’échelle conservent l’ordre des maisons, donc les mêmes coupures possibles : ils ne nuisent pas à un arbre, mais ne lui apportent rien.

**Option.** `make_base()` applique les deux par défaut. Pour un modèle à base d’arbres, on pourra les retirer avec `make_base(log=False, scaler="none")`.

## 7. Sélection des variables (stratégies)

Après la section 6, chaque maison est décrite par 227 colonnes, dont plusieurs mesurent la même chose : une somme et ses parties, une année et un âge, un équipement et son indicateur `Has…`. Nous comparons quatre façons de réduire ces colonnes avec la base, qui les garde toutes.

| Recette | Idée |
|---|---|
| Base | Garder les 227 colonnes |
| Corrélation | Garder les colonnes liées au prix, puis retirer les doublons statistiques |
| Causale | Retirer ce qui mesure une deuxième fois une même cause du prix |
| ACP | Remplacer les colonnes numériques par leurs premières composantes |
| Causale + ACP par blocs | Sélection causale, puis une composante par équipement |

**Protocole.** Chaque recette vient de `src/preprocessing/pipelines.py`. Elle est suivie de RidgeCV et évaluée avec la même validation croisée qu’aux sections 2 à 6. La sélection est apprise dans l’entraînement de chaque pli. Un modèle à base d’arbres choisit lui-même ses colonnes : pour lui, la sélection compte peu.

In [34]:
# ──────────────────────────────────────────────────────────────────────
# 7 · Protocole commun
# ──────────────────────────────────────────────────────────────────────
# Mêmes plis, même retrait ciblé et même RMSE log qu'aux sections 2 à 6 (cv_score).
raw_fit, raw_prices = remove_outliers(X, y)
memo = {}

def recipe_score(make, **options):
    """RMSE log d'une recette suivie de RidgeCV, en validation croisée."""
    key = (make.__name__, tuple(sorted(options.items())))
    if key not in memo:
        memo[key] = cv_score(lambda: make_pipeline(make(**options), RidgeCV(alphas=np.logspace(-1, 3, 30))))
    return memo[key]

def fitted_recipe(make, **options):
    """Recette ajustée sur train sans les deux ventes aberrantes."""
    return make(**options).fit(raw_fit, np.log(raw_prices))

def recipe_table(variants):
    """Colonnes produites et RMSE log de chaque variante {nom: (recette, options)}."""
    return pd.DataFrame(
        [[fitted_recipe(make, **options).transform(raw_fit).shape[1], recipe_score(make, **options)]
         for make, options in variants.values()],
        index=pd.Index(list(variants), name="Variante"), columns=["Colonnes", "RMSE log"]).round(4)

### 7.1 Sélection par corrélation

`CorrelationSelector` garde une colonne si son Spearman avec le prix atteint un seuil de **lien**. Il retire ensuite une colonne si son Spearman avec une colonne déjà gardée atteint un seuil de **redondance**.

**Décision.** Lien ≥ 0,05 et redondance 0,9 (`MIN_TARGET_CORRELATION`, `MAX_PAIR_CORRELATION`) : 154 colonnes sur 227.

**Justification.** Le tableau « Seuils de la sélection par corrélation » montre que la redondance 0,8 retire des colonnes proches mais utiles (0,1142), alors que 0,9 retire surtout des doublons (0,1120). Le lien 0,05 donne la même erreur que 0 avec 44 colonnes de moins; à 0,1, l’erreur monte (0,1134). Les 73 colonnes retirées comprennent 48 colonnes sans lien avec le prix et 25 doublons.

In [35]:
# ──────────────────────────────────────────────────────────────────────
# 7.1 Sélection par corrélation
# ──────────────────────────────────────────────────────────────────────
show_title("7.1 Sélection par corrélation")
show_title("Seuils de la sélection par corrélation")
display(recipe_table({
    f"Lien ≥ {min_target} · redondance {max_pair}".replace(".", ","):
        (make_correlation, {"min_target_corr": min_target, "max_pair_corr": max_pair})
    for min_target in (0, 0.05, 0.1) for max_pair in (0.8, 0.9)
}))
correlation_report = fitted_recipe(make_correlation)[-1].report_
show_title("Colonnes de la recette retenue (lien ≥ 0,05, redondance 0,9)")
display(correlation_report["Raison"].value_counts().rename("Colonnes").to_frame())

**7.1 Sélection par corrélation**

**Seuils de la sélection par corrélation**

,Colonnes,RMSE log
Variante,,
"Lien ≥ 0 · redondance 0,8",179,0.1142
"Lien ≥ 0 · redondance 0,9",198,0.1120
"Lien ≥ 0,05 · redondance 0,8",136,0.1142
"Lien ≥ 0,05 · redondance 0,9",154,0.1120
"Lien ≥ 0,1 · redondance 0,8",103,0.1153
"Lien ≥ 0,1 · redondance 0,9",119,0.1134


**Colonnes de la recette retenue (lien ≥ 0,05, redondance 0,9)**

,Colonnes
Raison,
"Liée au prix, non redondante",154
Lien faible avec le prix,48
Redondante,25


### 7.2 Sélection causale

`CausalSelector` retire, selon des règles fixées à l’avance, les colonnes qui mesurent une deuxième fois une même cause du prix :
1. un **doublon** : `YearBuilt` répète `AgeAtSale`;
2. une **conséquence** : `TotRmsAbvGrd` découle de `GrLivArea`;
3. une **surface comptée deux fois** : une somme gardée avec toutes ses parties.

Aucune colonne n’est retirée parce que son lien avec le prix est faible.

**Décision.** Garder les parties plutôt que les sommes (`CAUSAL_MODE`) : 10 colonnes retirées, 217 restantes.

**Justification.** Garder les parties donne 0,1126, contre 0,1142 en gardant les sommes : avec les parties, le modèle peut donner une valeur différente à un pi² de sous-sol, de rez-de-chaussée et d’étage. La sélection causale ne bat pas la base (0,1123), mais chaque cause garde une seule colonne, ce qui rend les coefficients lisibles.

In [36]:
# ──────────────────────────────────────────────────────────────────────
# 7.2 Sélection causale
# ──────────────────────────────────────────────────────────────────────
show_title("7.2 Sélection causale")
rule_of = {**{c: "1 · Doublon" for c in DUPLICATES}, **{c: "2 · Conséquence" for c in CONSEQUENCES}}
show_title("Colonnes retirées par la sélection causale")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([[rule_of.get(column, "3 · Surface comptée une fois"), column, reason]
                          for column, reason in causal_drops(CAUSAL_MODE).items()],
                         columns=["Règle", "Colonne retirée", "Raison"]).set_index(["Règle", "Colonne retirée"]))
show_title("Garder les sommes ou les parties")
display(recipe_table({"Sans sélection": (make_base, {}),
                      "Causale · sommes": (make_causal, {"mode": "totals"}),
                      "Causale · parties (choix)": (make_causal, {"mode": "parts"})}))

**7.2 Sélection causale**

**Colonnes retirées par la sélection causale**

Raison
Règle                        Colonne retirée                                                                                          
1 · Doublon                  YearBuilt                                          Doublon de AgeAtSale (même information, au signe près)
                             YearRemodAdd                               Doublon de YearsSinceRemodel (même information, au signe près)
                             GarageYrBlt      Doublon de l’âge : égale YearBuilt pour 79 % des garages et pour les maisons sans garage
2 · Conséquence              TotRmsAbvGrd                Conséquence de GrLivArea : la surface habitable détermine le nombre de pièces
                             GarageCars                 Conséquence de GarageArea : la surface du garage détermine le nombre de places
3 · Surface comptée une fois TotalSF                                                        Somme de TotalBsmtSF, 1stFlrSF et 2ndFlrSF
                             GrLivArea                                                     Somme de 1stFlrSF, 2ndFlrSF et LowQualFinSF
                             BsmtUnfSF                                       Reste du sous-sol : TotalBsmtSF − BsmtFinSF1 − BsmtFinSF2
                             TotalBath                                                     Somme des quatre colonnes de salles de bain
                             TotalPorchSF                                                          Somme des cinq colonnes de galeries

**Garder les sommes ou les parties**

,Colonnes,RMSE log
Variante,,
Sans sélection,227,0.1123
Causale · sommes,209,0.1142
Causale · parties (choix),217,0.1126


### 7.3 Analyse en composantes principales

`PCAReducer` remplace les 64 colonnes numériques standardisées par leurs premières composantes principales; les 163 colonnes one-hot restent telles quelles. `BlockPCA` applique plutôt l’ACP à de petits blocs de colonnes d’un même équipement (piscine, foyer, garage…), après la sélection causale : chaque bloc devient une seule composante.

**Décision.** Conserver 99 % de la variance (`PCA_VARIANCE`) : 49 composantes au lieu de 64 colonnes.

**Justification.** L’erreur baisse tant qu’on conserve de la variance : 0,1180 à 90 %, 0,1139 à 95 % et 0,1130 à 99 %. L’ACP classe les axes par variance sans regarder le prix, et les petites composantes portent encore de l’information. Le gain de place est modeste, et les composantes ne se lisent plus comme des caractéristiques de la maison.

In [37]:
# ──────────────────────────────────────────────────────────────────────
# 7.3 Analyse en composantes principales
# ──────────────────────────────────────────────────────────────────────
show_title("7.3 Analyse en composantes principales")
base_fitted = fitted_recipe(make_base)
numeric_names = base_fitted[-1].named_transformers_["numeric"].get_feature_names_out()
cumulative = np.cumsum(PCA().fit(base_fitted.transform(raw_fit)[numeric_names]).explained_variance_ratio_)
shares = [0.9, 0.95, 0.99]
pca_table = recipe_table({f"{share:.0%} de la variance": (make_pca, {"n_components": share}) for share in shares})
pca_table.insert(1, "Composantes", [int(np.searchsorted(cumulative, share) + 1) for share in shares])
show_title(f"Part de variance conservée ({len(numeric_names)} colonnes numériques au départ)")
display(pca_table)

**7.3 Analyse en composantes principales**

**Part de variance conservée (64 colonnes numériques au départ)**

,Colonnes,Composantes,RMSE log
Variante,,,
90% de la variance,193,30,0.1180
95% de la variance,200,37,0.1139
99% de la variance,212,49,0.1130


### 7.4 Comparaison et décision

Le tableau « 7.4 Comparaison des cinq recettes » réunit les recettes retenues aux sections 7.1 à 7.3.

**Décision. Recette de référence pour l’étape 05 : la base** (`make_base()`). Les cinq recettes se tiennent dans 0,0015 (de 0,1120 à 0,1135) : aucune ne fait nettement mieux que la base, qui ne perd aucune information.

- **Corrélation** (`make_correlation()`) : un peu meilleure (−0,0003) avec un tiers de colonnes en moins, mais ses seuils ont été choisis sur ces mêmes plis. Elle reste à tester à l’étape 5.
- **Causale** (`make_causal()`) : utile pour un modèle à interpréter.
- **ACP** et **causale + ACP par blocs** : moins bonnes ici.

In [38]:
# ──────────────────────────────────────────────────────────────────────
# 7.4 Comparaison des cinq recettes
# ──────────────────────────────────────────────────────────────────────
show_title("7.4 Comparaison des cinq recettes")
recipes_table = recipe_table({name: (make, {}) for name, make in RECIPES.items()})
recipes_table["Écart avec la base"] = recipes_table["RMSE log"] - recipes_table.loc["Base", "RMSE log"]
display(recipes_table.rename_axis("Recette").round(4))

**7.4 Comparaison des cinq recettes**

,Colonnes,RMSE log,Écart avec la base
Recette,,,
Base,227,0.1123,0.0000
Corrélation,154,0.1120,-0.0003
Causale,217,0.1126,0.0003
ACP,212,0.1130,0.0007
Causale + ACP par blocs,207,0.1135,0.0012


## 8. Reproduire le prétraitement

`src/preprocessing/pipelines.py` regroupe les décisions des sections 2 à 7 en recettes scikit-learn. Toute expérience de l’étape 5 doit respecter trois règles :

1. **La recette fait partie du modèle** : `make_pipeline(recette, modèle)`. Elle est ainsi réajustée sur l’entraînement de chaque pli, sans voir les maisons évaluées ni `test.csv`.
2. **Les deux ventes aberrantes** sont retirées de l’entraînement seulement, avant `fit`, avec `remove_outliers` (section 2.6). Un pipeline ne peut pas retirer des lignes de `y`.
3. **La cible est `log(SalePrice)`** : `TransformedTargetRegressor(func=np.log, inverse_func=np.exp)` entraîne sur le log et prédit en dollars. Le score est la RMSE log sur les ventes typiques (`cv_rmse`).

### 8.1 Recette selon la famille de modèles

| Famille | Exemples | Recette | Pourquoi |
|---|---|---|---|
| Arbres | XGBoost, Gradient Boosting, forêt aléatoire | `make_base(log=False, scaler="none")` | Un arbre coupe à des seuils : le log et la mise à l’échelle ne lui apportent rien (section 6.3) |
| Linéaires pénalisés | Ridge, Lasso, ElasticNet | `make_base()` | La pénalité exige une échelle commune; le log redresse les longues queues (section 6) |
| Noyaux et distances | Kernel Ridge, SVR, k plus proches voisins | `make_base()` | Les distances exigent une échelle commune (section 6.2) |
| Linéaire à interpréter | Régression linéaire sans pénalité | `make_causal()` | Une seule colonne par cause du prix (section 7.2) |
| Variante à tester | Tout modèle | `make_correlation()` | 154 colonnes au lieu de 227, à confirmer (section 7.4) |

Toutes les recettes produisent un DataFrame de colonnes numériques nommées, sans valeur absente. Le prétraitement ne fait aucun tirage aléatoire : avec les mêmes plis (`folds`, `random_state=42`), une expérience donne toujours le même résultat.

### 8.2 Code type

La cellule suivante est le modèle à copier. `with_preprocessing` assemble la recette, le modèle et la cible en log. `cross_validate` évalue en validation croisée, puis le modèle final s’ajuste sur tout train et prédit `test.csv`. L’exemple compare RidgeCV avec `make_base()` et XGBoost avec les deux recettes.

**Lecture.** Le tableau « RMSE log selon le modèle et la recette » montre que XGBoost obtient le même score (0,1158) avec ou sans log et mise à l’échelle, comme prévu à la section 6.3. Les réglages de XGBoost ne sont pas optimisés ici : ce sera le travail de l’étape 5.

In [45]:
# ──────────────────────────────────────────────────────────────────────
# 8.2 Code type
# ──────────────────────────────────────────────────────────────────────
show_title("8.2 Code type")
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor

def with_preprocessing(recipe, regressor):
    """Recette de l'étape 4, puis le modèle; entraîné sur log(SalePrice), prédit en dollars."""
    return TransformedTargetRegressor(make_pipeline(recipe, regressor), func=np.log, inverse_func=np.exp)

def cross_validate(model):
    """RMSE log hors pli (ventes typiques); le modèle complet est réajusté dans chaque pli."""
    predictions = pd.Series(np.nan, index=y.index)
    for fit_index, valid_index in folds.split(X):
        features, prices = remove_outliers(X.iloc[fit_index], y.iloc[fit_index])  # entraînement du pli seulement
        predictions.iloc[valid_index] = clone(model).fit(features, prices).predict(X.iloc[valid_index])
    return cv_rmse(y, predictions, X)

xgboost = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.7,
                       colsample_bytree=0.5, random_state=42, n_jobs=-1)
models = {
    ("RidgeCV", "make_base()"): with_preprocessing(make_base(), RidgeCV(alphas=np.logspace(-1, 3, 30))),
    ("XGBoost", 'make_base(log=False, scaler="none")'): with_preprocessing(make_base(log=False, scaler="none"), xgboost),
    ("XGBoost", "make_base()"): with_preprocessing(make_base(), xgboost),
}
show_title("RMSE log selon le modèle et la recette")
display(pd.Series({key: cross_validate(model) for key, model in models.items()}, name="RMSE log")
        .rename_axis(["Modèle", "Recette"]).round(4).to_frame())

# Modèle final : ajusté sur tout train sans les deux ventes aberrantes, puis prix de test.csv.
final = clone(models[("XGBoost", 'make_base(log=False, scaler="none")')]).fit(*remove_outliers(X, y))
test_prices = pd.Series(final.predict(test), index=test["Id"], name="SalePrice")
low, high = (f"{price:,.0f}".replace(",", " ") for price in (test_prices.min(), test_prices.max()))
print(f"{len(test_prices)} prix prédits pour test.csv, de {low} à {high} $")

**8.2 Code type**

**RMSE log selon le modèle et la recette**

RMSE log
Modèle  Recette                                      
RidgeCV make_base()                            0.1123
XGBoost make_base(log=False, scaler="none")    0.1158
        make_base()                            0.1158

1459 prix prédits pour test.csv, de 41 226 à 627 065 $


### 8.3 Vérification

Pour chaque recette, y compris l’option des arbres, ajustée sur train sans les deux ventes aberrantes :
- train et `test.csv` reçoivent les mêmes colonnes, dans le même ordre, même si `test.csv` contient des catégories absentes de train;
- aucune valeur n’est absente ni infinie;
- une maison de test reçoit les mêmes valeurs, seule ou dans un lot : `transform` n’apprend rien des maisons qu’il reçoit.

In [47]:
# ──────────────────────────────────────────────────────────────────────
# 8.3 Vérification
# ──────────────────────────────────────────────────────────────────────
show_title("8.3 Vérification")
recipes = {**RECIPES, 'Base · make_base(log=False, scaler="none")': lambda: make_base(log=False, scaler="none")}
rows = []
for name, make in recipes.items():
    recipe = make().fit(*remove_outliers(X, y))
    train_out, test_out = recipe.transform(X), recipe.transform(test)
    assert train_out.columns.equals(test_out.columns)
    assert np.isfinite(train_out.to_numpy(dtype=float)).all() and np.isfinite(test_out.to_numpy(dtype=float)).all()
    assert np.allclose(recipe.transform(test.iloc[[0]]).to_numpy(dtype=float), test_out.iloc[[0]].to_numpy(dtype=float))
    rows.append([name, train_out.shape[1]])
display(pd.DataFrame(rows, columns=["Recette", "Colonnes"]).set_index("Recette"))

**8.3 Vérification**

,Colonnes
Recette,
Base,227
Corrélation,154
Causale,217
ACP,212
Causale + ACP par blocs,207
"Base · make_base(log=False, scaler=""none"")",227


## 9. Bilan

| Étape | Choix retenu | Preuve principale (RMSE log, ventes typiques) | Section |
|---|---|---|---|
| Évaluation | Scores calculés sans les ventes 524 et 1299 (`cv_rmse`) | Ces deux ventes dominent la RMSE de toutes les ventes | 1 |
| Nettoyage | Codes harmonisés; `Id` et `Utilities` retirées; retrait ciblé des deux ventes aberrantes, à l’entraînement seulement | 0,1137 contre 0,1183 en les gardant | 2 |
| Valeurs manquantes | Équipement absent → `None` ou 0; année du garage → `YearBuilt`; façade → médiane du quartier; repli sur les statistiques de train | 79 % des garages ont l’année de la maison | 3 |
| Création | 14 colonnes ajoutées sans retrait | Spearman de `TotalSF` 0,82 contre 0,73 pour `GrLivArea`; `QualxArea` 0,85 contre 0,81 pour `OverallQual` | 4 |
| Encodage | Ordinal pour 15 échelles, one-hot pour 28 variables nominales, catégories de moins de 10 maisons regroupées | Regroupement : 36 colonnes de moins pour la même RMSE (0,1138) | 5 |
| Forme et échelle | `log1p` sur 10 colonnes à longue queue; standardisation des 64 colonnes numériques | 0,1123 contre 0,1146 sans log | 6 |
| Sélection | Base sans sélection; corrélation à tester à l’étape 5 | Les cinq recettes dans 0,0015 | 7 |
| Pipelines | Cinq recettes dans `src/preprocessing/pipelines.py`, ajustées sur l’entraînement seulement | Vérifications de la section 8 | 8 |

**Ce que reçoit l’étape 5.** `make_base()` transforme une maison brute en 227 colonnes numériques sans valeur absente. Avec RidgeCV, elle donne une RMSE log de 0,1123 sur les ventes typiques : c’est la référence à battre.

**Limites.**
- **Écarts petits.** Plusieurs décisions reposent sur des écarts de 0,001 ou moins, avec un seul découpage en plis : un autre découpage pourrait changer leur ordre.
- **Choisi pour Ridge.** Le log, la standardisation et la sélection ont été comparés avec un modèle linéaire. Un modèle à base d’arbres n’est pas sensible au log ni à la mise à l’échelle : `make_base(log=False, scaler="none")` les retire (section 6.3). Il pourrait aussi préférer d’autres choix.
- **Mêmes plis pour choisir et mesurer.** Les scores des options retenues sont donc légèrement optimistes. Aucune partition n’est réservée (section 1) : seule la soumission Kaggle donnera une mesure indépendante.
- **Ventes typiques.** Le score ignore les deux ventes aberrantes. `test.csv` contient aussi une très grande maison (Id 2550, 5 095 pi²). Si sa vente ressemble aux deux ventes aberrantes de train, son erreur sera grande.